In [ ]:
# Quick test - basic imports
import sys
print(f"Python version: {sys.version}")
print("Basic import successful!")

## 1. Import Required Libraries

In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Scikit-learn utilities
import sklearn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, chi2, RFE, mutual_info_classif
from sklearn.decomposition import PCA

# Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# Advanced ML libraries (optional - install libomp via brew if needed)
try:
    import xgboost as xgb
    print("✓ XGBoost imported successfully")
except Exception as e:
    print(f"⚠ XGBoost not available. Run: brew install libomp")
    xgb = None

try:
    import lightgbm as lgb
    print("✓ LightGBM imported successfully")
except Exception as e:
    print(f"⚠ LightGBM not available: {str(e)[:50]}...")
    lgb = None

try:
    from catboost import CatBoostClassifier
    print("✓ CatBoost imported successfully")
except Exception as e:
    print(f"⚠ CatBoost not available: {str(e)[:50]}...")
    CatBoostClassifier = None

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, auc,
    confusion_matrix, classification_report, matthews_corrcoef
)

# Handling imbalanced data
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek, SMOTEENN

# Model interpretation
try:
    import shap
    print("✓ SHAP imported successfully")
except Exception as e:
    print(f"⚠ SHAP not available: {str(e)[:50]}...")
    shap = None

try:
    from lime import lime_tabular
    print("✓ LIME imported successfully")
except Exception as e:
    print(f"⚠ LIME not available: {str(e)[:50]}...")
    lime_tabular = None

# Model persistence
import joblib
import pickle

# Statistical analysis
from scipy import stats
from scipy.stats import zscore

# Date handling
from datetime import datetime

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("\n" + "="*60)
print("✓ Core libraries imported successfully!")
print("="*60)
print(f"Python version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print("="*60)

## 2. Define File Paths and Configuration

In [ ]:
# Base directory
BASE_DIR = Path('/Users/roya/Desktop/Thesis')
DATA_DIR = BASE_DIR / 'Data set'
OUTPUT_DIR = BASE_DIR / 'Thesis_Project' / 'ML'

# Data file paths
PART_B_FILE = DATA_DIR / 'Part B' / '2023' / 'MUP_PHY_R25_P05_V20_D23_Prov_Svc.csv'
PART_D_FILE = DATA_DIR / 'Medicare Part D Prescribers - by Provider and Drug' / '2023' / 'MUP_DPR_RY25_P04_V10_DY23_NPIBN.csv'
OIG_FILE = DATA_DIR / 'OIG.csv'

# Create output directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'models').mkdir(exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(exist_ok=True)
(OUTPUT_DIR / 'reports').mkdir(exist_ok=True)

print("✓ File paths configured")
print(f"Base Directory: {BASE_DIR}")
print(f"Data Directory: {DATA_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")

## 3. Data Loading & Initial Exploration

### 3.1 Load Medicare Part B Data

In [ ]:
# Load Part B data (Provider Services)
print("Loading Medicare Part B 2023 data...")
part_b_df = pd.read_csv(PART_B_FILE, low_memory=False)

print(f"Part B Data Shape: {part_b_df.shape}")
print(f"\nMemory Usage: {part_b_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
part_b_df.head()

### 3.2

In [ ]:
# Filter to include only U.S. states and territories

us_states = ['AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA', 
             'HI', 'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD', 
             'MA', 'MI', 'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ', 
             'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 
             'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY', 
             'DC', 'PR', 'VI', 'GU', 'AS', 'MP']

before_filter_b = len(part_b_df)
part_b_df = part_b_df[part_b_df['Rndrng_Prvdr_State_Abrvtn'].isin(us_states)]
after_filter_b = len(part_b_df)
removed_b = before_filter_b - after_filter_b

print(f"Part B: Removed {removed_b:,} non-U.S. records ({removed_b/before_filter_b*100:.2f}%)")
print(f"Part B: Remaining records: {after_filter_b:,}")
print(f"Part B Data Shape after filtering: {part_b_df.shape}")

### 3.3

In [ ]:
# Check if each provider has consistent location information
print("\n" + "="*80)
print("PROVIDER LOCATION CONSISTENCY CHECK - PART B")
print("="*80)

# Group by NPI and count unique values for State, Zipcode, and RUCA
provider_location_check_b = part_b_df.groupby('Rndrng_NPI').agg({
    'Rndrng_Prvdr_State_Abrvtn': 'nunique',
    'Rndrng_Prvdr_Zip5': 'nunique',
    'Rndrng_Prvdr_RUCA': 'nunique'
}).rename(columns={
    'Rndrng_Prvdr_State_Abrvtn': 'Unique_States',
    'Rndrng_Prvdr_Zip5': 'Unique_Zipcodes',
    'Rndrng_Prvdr_RUCA': 'Unique_RUCA'
})

# Count providers with single vs multiple values
total_providers = len(provider_location_check_b)
single_state = (provider_location_check_b['Unique_States'] == 1).sum()
multiple_states = (provider_location_check_b['Unique_States'] > 1).sum()

single_zip = (provider_location_check_b['Unique_Zipcodes'] == 1).sum()
multiple_zip = (provider_location_check_b['Unique_Zipcodes'] > 1).sum()

single_ruca = (provider_location_check_b['Unique_RUCA'] == 1).sum()
multiple_ruca = (provider_location_check_b['Unique_RUCA'] > 1).sum()

print(f"\nTotal unique providers: {total_providers:,}")
print(f"\n{'Field':<20} {'Single Value':<20} {'Multiple Values':<20}")
print("-" * 60)
print(f"{'State':<20} {single_state:,} ({single_state/total_providers*100:.2f}%)    {multiple_states:,} ({multiple_states/total_providers*100:.2f}%)")
print(f"{'Zipcode':<20} {single_zip:,} ({single_zip/total_providers*100:.2f}%)    {multiple_zip:,} ({multiple_zip/total_providers*100:.2f}%)")
print(f"{'RUCA Code':<20} {single_ruca:,} ({single_ruca/total_providers*100:.2f}%)    {multiple_ruca:,} ({multiple_ruca/total_providers*100:.2f}%)")

# Display Min and Max statistics for each unique column
print("\n" + "="*80)
print("MIN AND MAX STATISTICS FOR UNIQUE VALUES")
print("="*80)
print(f"\n{'Field':<20} {'Min':<15} {'Max':<15} {'Mean':<15}")
print("-" * 65)
print(f"{'Unique_States':<20} {provider_location_check_b['Unique_States'].min():<15} {provider_location_check_b['Unique_States'].max():<15} {provider_location_check_b['Unique_States'].mean():<15.2f}")
print(f"{'Unique_Zipcodes':<20} {provider_location_check_b['Unique_Zipcodes'].min():<15} {provider_location_check_b['Unique_Zipcodes'].max():<15} {provider_location_check_b['Unique_Zipcodes'].mean():<15.2f}")
print(f"{'Unique_RUCA':<20} {provider_location_check_b['Unique_RUCA'].min():<15} {provider_location_check_b['Unique_RUCA'].max():<15} {provider_location_check_b['Unique_RUCA'].mean():<15.2f}")

# Create detailed table for each provider
print("\n" + "="*80)
print("DETAILED PROVIDER LOCATION TABLE")
print("="*80)

# Add flag for consistency
provider_location_check_b['Has_Multiple_Locations'] = (
    (provider_location_check_b['Unique_States'] > 1) |
    (provider_location_check_b['Unique_Zipcodes'] > 1) |
    (provider_location_check_b['Unique_RUCA'] > 1)
)

# Display the table
print(f"\nShowing all {len(provider_location_check_b):,} providers:")
display(provider_location_check_b)

# Show providers with multiple locations
if multiple_states > 0 or multiple_zip > 0 or multiple_ruca > 0:
    print(f"\n{'='*80}")
    print("PROVIDERS WITH INCONSISTENT LOCATIONS")
    print("="*80)
    multi_location_providers = provider_location_check_b[
        provider_location_check_b['Has_Multiple_Locations']
    ]
    print(f"\nTotal providers with multiple locations: {len(multi_location_providers):,}")
    display(multi_location_providers)

print("="*80)

In [ ]:
print(part_b_df.columns)

remove = ['Rndrng_Prvdr_Last_Org_Name', 'Rndrng_Prvdr_First_Name', 'Rndrng_Prvdr_MI', 'Rndrng_Prvdr_Crdntls', 'Rndrng_Prvdr_St1', 'Rndrng_Prvdr_St2', 'Rndrng_Prvdr_City', 'Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_RUCA_Desc', 'Rndrng_Prvdr_Cntry', 'Rndrng_Prvdr_Mdcr_Prtcptg_Ind', 'HCPCS_Desc', ]

keep = [ 'Rndrng_NPI', 'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_Zip5', Rndrng_Prvdr_RUCA', 'Rndrng_Prvdr_Cntry', 'Rndrng_Prvdr_Ent_Cd', '', 'Rndrng_Prvdr_Type', 'HCPCS_Cd', 'HCPCS_Drug_Ind', 'Place_Of_Srvc', 'Tot_Benes',	'Tot_Srvcs', 'Tot_Bene_Day_Srvcs', 'Avg_Sbmtd_Chrg',	'Avg_Mdcr_Alowd_Amt', 	'Avg_Mdcr_Pymt_Amt', 	'Avg_Mdcr_Stdzd_Amt'   ]

In [ ]:
part_b_df['Rndrng_Prvdr_Ent_Cd'].unique()

In [ ]:
part_b_df['Rndrng_Prvdr_Cntry'].unique()

In [ ]:
part_b_df['HCPCS_Drug_Ind'].unique()

In [ ]:
part_b_df['Rndrng_Prvdr_Crdntls'].unique()

In [ ]:
part_b_df['Rndrng_Prvdr_Mdcr_Prtcptg_Ind'].unique()

In [ ]:
part_b_df['Rndrng_Prvdr_RUCA'].unique()

### 3.4

In [ ]:
provider_summary = part_b_df.groupby('Rndrng_NPI').agg({
    'Rndrng_Prvdr_Type': lambda x: list(x.unique()),
    'HCPCS_Cd': lambda x: list(x.unique()),
    'HCPCS_Drug_Ind': lambda x: list(x.unique())
}).reset_index()

provider_summary['Num_Unique_Provider_Types'] = provider_summary['Rndrng_Prvdr_Type'].apply(len)
provider_summary['Num_Unique_HCPCS_Codes'] = provider_summary['HCPCS_Cd'].apply(len)
provider_summary['Num_Unique_HCPCS_Drug_Ind'] = provider_summary['HCPCS_Drug_Ind'].apply(len)

provider_summary['Provider_Types'] = provider_summary['Rndrng_Prvdr_Type'].apply(lambda x: ', '.join(map(str, x)))
provider_summary['HCPCS_Codes'] = provider_summary['HCPCS_Cd'].apply(lambda x: ', '.join(map(str, x)))
provider_summary['HCPCS_Drug_Ind_List'] = provider_summary['HCPCS_Drug_Ind'].apply(lambda x: ', '.join(map(str, x)))

provider_summary_display = provider_summary[[
    'Rndrng_NPI',
    'Num_Unique_Provider_Types',
    'Num_Unique_HCPCS_Codes',
    'Num_Unique_HCPCS_Drug_Ind',
    'Provider_Types',
    'HCPCS_Codes',
    'HCPCS_Drug_Ind_List'
]]

print(f"\nTotal number of unique providers: {len(provider_summary_display)}")
print(f"\nFirst 20 providers:")
print(provider_summary_display.head(20).to_string(index=False))


print("SUMMARY STATISTICS")
print(f"Average provider types per provider: {provider_summary_display['Num_Unique_Provider_Types'].mean():.2f}")
print(f"Average HCPCS codes per provider: {provider_summary_display['Num_Unique_HCPCS_Codes'].mean():.2f}")
print(f"Average HCPCS Drug Ind per provider: {provider_summary_display['Num_Unique_HCPCS_Drug_Ind'].mean():.2f}")
print(f"Max provider types for a single provider: {provider_summary_display['Num_Unique_Provider_Types'].max()}")
print(f"Max HCPCS codes for a single provider: {provider_summary_display['Num_Unique_HCPCS_Codes'].max()}")
print(f"Max HCPCS Drug Ind for a single provider: {provider_summary_display['Num_Unique_HCPCS_Drug_Ind'].max()}")


provider_summary_display

In [ ]:
part_b_df['Place_Of_Srvc'].unique()

In [ ]:
part_b_df['HCPCS_Drug_Ind'].unique()

In [ ]:
# Basic information about Part B data
print("Part B Data Info:")
print("="*50)
part_b_df.info()

In [ ]:
# Descriptive statistics for Part B
part_b_df.describe()

### 3.2 Load Medicare Part D Data

### 3.2.1

In [ ]:
# Load Part D data (Prescription Drugs)
print("Loading Medicare Part D 2023 data...")
part_d_df = pd.read_csv(PART_D_FILE, low_memory=False)

print(f"Part D Data Shape: {part_d_df.shape}")
print(f"\nMemory Usage: {part_d_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

part_d_df.head()

### 3.2.2

In [ ]:

# Filter Part D data
before_filter_d = len(part_d_df)
part_d_df = part_d_df[part_d_df['Prscrbr_State_Abrvtn'].isin(us_states)]
after_filter_d = len(part_d_df)
removed_d = before_filter_d - after_filter_d

print(f"Part D: Removed {removed_d:,} non-U.S. records ({removed_d/before_filter_d*100:.2f}%)")
print(f"Part D: Remaining records: {after_filter_d:,}")
print(f"Part D Data Shape after filtering: {part_d_df.shape}")

### 3.2.3

In [ ]:
# Check if each provider has consistent location information
print("\n" + "="*80)
print("PROVIDER LOCATION CONSISTENCY CHECK - PART D")
print("="*80)

# Group by NPI and count unique values for State and Zipcode (Part D doesn't have RUCA)
provider_location_check_d = part_d_df.groupby('Prscrbr_NPI').agg({
    'Prscrbr_State_Abrvtn': 'nunique',
    'Prscrbr_State_FIPS': 'nunique'
}).rename(columns={
    'Prscrbr_State_Abrvtn': 'Unique_States',
    'Prscrbr_State_FIPS': 'Unique_State_FIPS'
})

# Count providers with single vs multiple values
total_providers_d = len(provider_location_check_d)
single_state_d = (provider_location_check_d['Unique_States'] == 1).sum()
multiple_states_d = (provider_location_check_d['Unique_States'] > 1).sum()

single_fips_d = (provider_location_check_d['Unique_State_FIPS'] == 1).sum()
multiple_fips_d = (provider_location_check_d['Unique_State_FIPS'] > 1).sum()

print(f"\nTotal unique prescribers: {total_providers_d:,}")
print(f"\n{'Field':<20} {'Single Value':<20} {'Multiple Values':<20}")
print("-" * 60)
print(f"{'State':<20} {single_state_d:,} ({single_state_d/total_providers_d*100:.2f}%)    {multiple_states_d:,} ({multiple_states_d/total_providers_d*100:.2f}%)")
print(f"{'State FIPS':<20} {single_fips_d:,} ({single_fips_d/total_providers_d*100:.2f}%)    {multiple_fips_d:,} ({multiple_fips_d/total_providers_d*100:.2f}%)")

# Display Min and Max statistics for each unique column
print("\n" + "="*80)
print("MIN AND MAX STATISTICS FOR UNIQUE VALUES")
print("="*80)
print(f"\n{'Field':<20} {'Min':<15} {'Max':<15} {'Mean':<15}")
print("-" * 65)
print(f"{'Unique_States':<20} {provider_location_check_d['Unique_States'].min():<15} {provider_location_check_d['Unique_States'].max():<15} {provider_location_check_d['Unique_States'].mean():<15.2f}")
print(f"{'Unique_State_FIPS':<20} {provider_location_check_d['Unique_State_FIPS'].min():<15} {provider_location_check_d['Unique_State_FIPS'].max():<15} {provider_location_check_d['Unique_State_FIPS'].mean():<15.2f}")

# Create detailed table for each provider
print("\n" + "="*80)
print("DETAILED PROVIDER LOCATION TABLE")
print("="*80)

# Add flag for consistency
provider_location_check_d['Has_Multiple_Locations'] = (
    (provider_location_check_d['Unique_States'] > 1) |
    (provider_location_check_d['Unique_State_FIPS'] > 1)
)

# Display the table
print(f"\nShowing all {len(provider_location_check_d):,} prescribers:")
display(provider_location_check_d)

# Show providers with multiple locations
if multiple_states_d > 0 or multiple_fips_d > 0:
    print(f"\n{'='*80}")
    print("PRESCRIBERS WITH INCONSISTENT LOCATIONS")
    print("="*80)
    multi_location_prescribers = provider_location_check_d[
        provider_location_check_d['Has_Multiple_Locations']
    ]
    print(f"\nTotal prescribers with multiple locations: {len(multi_location_prescribers):,}")
    display(multi_location_prescribers)

print("="*80)

In [ ]:
print(part_d_df.columns)

remove = ['Prscrbr_Last_Org_Name', 'Prscrbr_First_Name', 'Prscrbr_City', 'Prscrbr_State_FIPS', 'Prscrbr_Type_Src',  'GE65_Sprsn_Flag', 'GE65_Bene_Sprsn_Flag', 'GE65_Tot_Clms', 'GE65_Tot_30day_Fills', 'GE65_Tot_Drug_Cst', 'GE65_Tot_Day_Suply', 'GE65_Tot_Benes']

keep = ['Prscrbr_NPI', , 'Prscrbr_State_Abrvtn', 'Prscrbr_Type', 'Brnd_Name', 'Gnrc_Name','Tot_Clms', 'Tot_30day_Fills', 'Tot_Day_Suply', 'Tot_Drug_Cst','Tot_Benes']

Tot_30day_Fills': 'sum',  # How many monthly prescriptions the provider wrote

'Tot_Day_Suply': 'sum',  # Total number of days of medication given to patients

'Tot_Benes': 'sum',  # Number of different Medicare patients who received this drug from this provider(note: 0 means suppressed <11)
    
 how much money
'Tot_Drug_Cst': ['sum'],  # total money spent at pharmacies because of this provider’s prescriptions

### 3.2.5

In [ ]:
print("PROVIDER SUMMARY TABLE - PART D")

provider_summary_d = part_d_df.groupby('Prscrbr_NPI').agg({
    'Prscrbr_Type': lambda x: list(x.unique()), # Unique = Duplicate, lambda = summerize the function
    'Brnd_Name': lambda x: list(x.unique()),
    'Gnrc_Name': lambda x: list(x.unique())
}).reset_index() # groupby moves NPI to the left margin, brings it back into the table 

# In this part of the code I create a new column that counts the number of unique items stored inside each row’s list/set column.
provider_summary_d['Num_Unique_Provider_Types'] = provider_summary_d['Prscrbr_Type'].apply(len)
provider_summary_d['Num_Unique_Brnd_Names'] = provider_summary_d['Brnd_Name'].apply(len)
provider_summary_d['Num_Unique_Gnrc_Names'] = provider_summary_d['Gnrc_Name'].apply(len)

# Convert lists to strings for better display
provider_summary_d['Provider_Types'] = provider_summary_d['Prscrbr_Type'].apply(lambda x: ', '.join(map(str, x)))
provider_summary_d['Brand_Names'] = provider_summary_d['Brnd_Name'].apply(lambda x: ', '.join(map(str, x)))
provider_summary_d['Generic_Names'] = provider_summary_d['Gnrc_Name'].apply(lambda x: ', '.join(map(str, x)))

# Create final display table with reordered columns
provider_summary_d_display = provider_summary_d[[
    'Prscrbr_NPI',
    'Num_Unique_Provider_Types',
    'Num_Unique_Brnd_Names',
    'Num_Unique_Gnrc_Names',
    'Provider_Types',
    'Brand_Names',
    'Generic_Names'
]]

print(f"\nTotal number of unique providers: {len(provider_summary_d_display)}")
print(f"\nFirst 20 providers:")
print(provider_summary_d_display.head(20).to_string(index=False))

print("SUMMARY STATISTICS - PART D")
print(f"Average provider types per provider: {provider_summary_d_display['Num_Unique_Provider_Types'].mean():.2f}")
print(f"Average brand names per provider: {provider_summary_d_display['Num_Unique_Brnd_Names'].mean():.2f}")
print(f"Average generic names per provider: {provider_summary_d_display['Num_Unique_Gnrc_Names'].mean():.2f}")
print(f"Max provider types for a single provider: {provider_summary_d_display['Num_Unique_Provider_Types'].max()}")
print(f"Max brand names for a single provider: {provider_summary_d_display['Num_Unique_Brnd_Names'].max()}")
print(f"Max generic names for a single provider: {provider_summary_d_display['Num_Unique_Gnrc_Names'].max()}")

# Show the full table (optional - can be commented out for large datasets)
provider_summary_d_display

In [ ]:
part_d_df['Prscrbr_State_Abrvtn'].unique()

In [ ]:
part_d_df['Prscrbr_Type_Src'].unique()

In [ ]:
# Basic information about Part D data
print("Part D Data Info:")
print("="*50)
part_d_df.info()

### 3.3 Load OIG Exclusion List

In [ ]:
# Load OIG exclusion list
print("Loading OIG Exclusion List...")
oig_df = pd.read_csv(OIG_FILE, low_memory=False)

print(f"OIG Data Shape: {oig_df.shape}")
print(f"\nMemory Usage: {oig_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

oig_df.head()

In [ ]:
print(oig_df.columns)

In [ ]:
# Examine OIG data structure
print("OIG Data Info:")
oig_df.info()

print("Unique values in key columns:")
print(f"Unique Last Names: {oig_df['LASTNAME'].nunique() if 'LASTNAME' in oig_df.columns else 'N/A'}")
print(f"Unique States: {oig_df['STATE'].nunique() if 'STATE' in oig_df.columns else 'N/A'}")

## 4. Data Quality Assessment

### 4.1 Missing Values Analysis

In [ ]:
def analyze_missing_values(df, df_name):
    print(f"Missing Values Analysis: {df_name}")

    missing_values = ['', ' ', '  ', 'nan', 'NaN', 'NA', 'N/A', 'null', 'NULL', 'None', 'none', '<NA>']
    df_temp = df.replace(missing_values, pd.NA)
    
    missing_count = df_temp.isnull().sum()
    missing_percent = (missing_count / len(df_temp)) * 100
    
    missing_df = pd.DataFrame({
        'Column': df_temp.columns,
        'Missing_Count': missing_count.values,
        'Missing_Percent': missing_percent.values
    })
    
    missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values(
        'Missing_Percent', ascending=False
    ).reset_index(drop=True)
    
    if len(missing_df) == 0:
        print("✓ No missing values found!")
    else:
        print(f"\nColumns with missing values: {len(missing_df)}")
        print(missing_df)
        
        if len(missing_df) <= 20:
            fig = px.bar(
                missing_df,
                x='Column',
                y='Missing_Percent',
                title=f'Missing Values in {df_name}',
                labels={'Missing_Percent': 'Missing %'},
                text='Missing_Percent'
            )
            fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
            fig.update_layout(xaxis_tickangle=-45, height=500)
            fig.show()
    
    return missing_df

missing_part_b = analyze_missing_values(part_b_df, 'Medicare Part B 2023')
missing_part_d = analyze_missing_values(part_d_df, 'Medicare Part D 2023')
missing_oig = analyze_missing_values(oig_df, 'OIG Exclusion List')

### 4.2 Duplicate Records Check

In [ ]:
def check_duplicates(df, df_name):
    print(f"\nDuplicate Analysis: {df_name}")

    full_duplicates = df.duplicated().sum()
    print(f"Full duplicate rows: {full_duplicates} ({full_duplicates/len(df)*100:.2f}%)")
    
    return full_duplicates

dup_part_b = check_duplicates(part_b_df, 'Part B')
dup_part_d = check_duplicates(part_d_df, 'Part D')
dup_oig = check_duplicates(oig_df, 'OIG')

## 5. Data Preprocessing & Cleaning


In [ ]:
def basic_data_cleaning(df, df_name):
    
    df_clean = df.copy()
    initial_rows = len(df_clean)
    
    # Step 1: Remove exact duplicates
    before_dedup = len(df_clean)
    df_clean = df_clean.drop_duplicates()
    after_dedup = len(df_clean)
    removed_dups = before_dedup - after_dedup
    print(f"1. Removed duplicates: {removed_dups:,} rows")
    
    # Step 2: Standardize NPI format (if NPI column exists)
    npi_cols = [col for col in df_clean.columns if 'NPI' in col.upper()]
    for npi_col in npi_cols:
        if df_clean[npi_col].dtype == 'object':
            # Convert to string, remove whitespace
            df_clean[npi_col] = df_clean[npi_col].astype(str).str.strip()
            # Remove non-numeric NPIs (keep NaN as-is)
            valid_npi = df_clean[npi_col].str.isnumeric() | df_clean[npi_col].isna()
            invalid_count = (~valid_npi).sum()
            if invalid_count > 0:
                df_clean = df_clean[valid_npi]
                print(f"2. Removed invalid NPIs in {npi_col}: {invalid_count:,} rows")
    
    # Step 3: Remove rows with completely missing data (all NaN)
    all_nan_rows = df_clean.isnull().all(axis=1).sum()
    if all_nan_rows > 0:
        df_clean = df_clean.dropna(how='all')
        print(f"3. Removed completely empty rows: {all_nan_rows:,}")
    
    final_rows = len(df_clean)
    total_removed = initial_rows - final_rows
    missing_count = df_clean.isnull().sum().sum()
    
    print(f"\nCLEANING SUMMARY:")
    print(f"  Initial rows:     {initial_rows:,}")
    print(f"  Final rows:       {final_rows:,}")
    print(f"  Rows removed:     {total_removed:,} ({total_removed/initial_rows*100:.2f}%)")
    print(f"  Missing values:   {missing_count:,} (PRESERVED for post-aggregation imputation)")
    print("="*80)
    print()
    
    return df_clean


In [ ]:
print("Applying basic data cleaning (no imputation yet)...")
print()

part_b_clean = basic_data_cleaning(part_b_df, "Part B - Provider Services")
part_d_clean = basic_data_cleaning(part_d_df, "Part D - Prescription Drugs")
oig_clean = basic_data_cleaning(oig_df, "OIG - Exclusion List")


print("✓ Basic cleaning complete! Missing values preserved for post-aggregation imputation.")


## 6. Provider-Level Aggregation


### 6.1 Create Provider-Level Aggregations - Part B

We'll aggregate data at the provider (NPI) level to create features.

In [ ]:
print("Creating provider-level aggregations from Part B data...")

part_b_clean['Weighted_Payment'] = part_b_clean['Tot_Srvcs'] * part_b_clean['Avg_Mdcr_Pymt_Amt']

# Aggregate Part B data by provider (NPI)
part_b_agg = part_b_clean.groupby('Rndrng_NPI').agg({
    # Provider information (first value since it should be same for each NPI)
    'Rndrng_Prvdr_Last_Org_Name': 'first',
    'Rndrng_Prvdr_First_Name': 'first',
    'Rndrng_Prvdr_State_Abrvtn': 'first',
    'Rndrng_Prvdr_Zip5': 'first',
    'Rndrng_Prvdr_Type': 'first',  # Derived from the provider specialty code reported on the claim.
    'Rndrng_Prvdr_Ent_Cd': 'first',  # Entity code (Individual vs Organization)
    'Rndrng_Prvdr_RUCA': 'first',  # Rural-Urban Commuting Area code

    # Volume metrics - Total counts
    'Tot_Benes': 'mean',  # Average distinct Medicare beneficiaries per service line
    'Tot_Srvcs': 'sum',
    'Tot_Bene_Day_Srvcs': 'sum',

    # Financial metrics - Average amounts per service
    'Avg_Sbmtd_Chrg': ['mean', 'max'],      # Charges that the provider submitted
    'Avg_Mdcr_Alowd_Amt': ['mean', 'max'],  # Medicare allowed amount
    'Avg_Mdcr_Pymt_Amt': 'mean',            # Medicare payment after deductible/coinsurance
    'Avg_Mdcr_Stdzd_Amt': 'mean',           # Standardized Medicare payment (geography-adjusted)
    'Weighted_Payment': 'sum',              # Numerator: sum(Avg_Mdcr_Pymt_Amt × Tot_Srvcs) — divided below

    # Service diversity and complexity
    'HCPCS_Cd': 'nunique',          # Number of unique procedure/service codes
    'HCPCS_Drug_Ind': 'nunique',    # Number of unique drug indicators (drug vs non-drug)
    'Place_Of_Srvc': 'nunique',     # Number of unique service places (facility vs non-facility)
}).reset_index()

part_b_agg.columns = ['_'.join(col).strip('_') if isinstance(col, tuple) else col for col in part_b_agg.columns.values]

# Rename columns for clarity
column_mapping = {
    'Rndrng_NPI': 'NPI',
    'Rndrng_Prvdr_Last_Org_Name_first': 'Provider_Name',
    'Rndrng_Prvdr_First_Name_first': 'Provider_FirstName',
    'Rndrng_Prvdr_State_Abrvtn_first': 'State',
    'Rndrng_Prvdr_Zip5_first': 'Zipcode',
    'Rndrng_Prvdr_Type_first': 'Provider_Type',
    'Rndrng_Prvdr_Ent_Cd_first': 'Entity_Code',
    'Rndrng_Prvdr_RUCA_first': 'RUCA_Code',
    # Volume
    'Tot_Benes_mean': 'Total_Beneficiaries_B',
    'Tot_Srvcs_sum': 'Total_Services_B',
    'Tot_Bene_Day_Srvcs_sum': 'Total_BeneDay_Services_B',
    # Financial
    'Avg_Sbmtd_Chrg_mean': 'Avg_Submitted_Charge_B',
    'Avg_Sbmtd_Chrg_max': 'Max_Submitted_Charge_B',
    'Avg_Mdcr_Alowd_Amt_mean': 'Avg_Medicare_Allowed_B',
    'Avg_Mdcr_Alowd_Amt_max': 'Max_Medicare_Allowed_B',
    'Avg_Mdcr_Pymt_Amt_mean': 'Avg_Medicare_Payment_B',
    'Avg_Mdcr_Stdzd_Amt_mean': 'Avg_Medicare_Standardized_B',
    'Weighted_Payment_sum': 'Weighted_Avg_Payment_B',  # Will be divided by Total_Services_B below
    # Diversity
    'HCPCS_Cd_nunique': 'Unique_HCPCS_B',
    'HCPCS_Drug_Ind_nunique': 'Service_Type_Diversity_B',  # 1=one type (Drug/Non-Drug), 2=both
    'Place_Of_Srvc_nunique': 'Unique_Service_Places_B',
}
part_b_agg.rename(columns=column_mapping, inplace=True)


part_b_agg['Weighted_Avg_Payment_B'] = (
    part_b_agg['Weighted_Avg_Payment_B'] / part_b_agg['Total_Services_B']
)

print(f"✓ Part B aggregated to {len(part_b_agg):,} unique providers")
print(f"  Columns: {part_b_agg.shape[1]}")
print(f"  Features: {part_b_agg.shape[1] - 7} numeric + 7 descriptive")
part_b_agg.head()


In [ ]:

# Unique_HCPCS_B — number of unique HCPCS procedure codes billed per provider
min_val = part_b_agg['Unique_HCPCS_B'].min()
max_val = part_b_agg['Unique_HCPCS_B'].max()

print(f"Unique_HCPCS_B (unique HCPCS codes per provider):")
print(f"  Min : {min_val:,}")
print(f"  Max : {max_val:,}")
print(f"\nProvider(s) with Min unique HCPCS codes ({min_val}):")
print(part_b_agg.loc[part_b_agg['Unique_HCPCS_B'] == min_val,
                     ['NPI', 'Provider_Name', 'Provider_Type', 'Unique_HCPCS_B']].head())
print(f"\nProvider(s) with Max unique HCPCS codes ({max_val}):")
print(part_b_agg.loc[part_b_agg['Unique_HCPCS_B'] == max_val,
                     ['NPI', 'Provider_Name', 'Provider_Type', 'Unique_HCPCS_B']].head())


### 6.2 Create Provider-Level Aggregations - Part D

In [ ]:
print("\nCreating provider-level aggregations from Part D data...")

# Aggregate Part D data by prescriber (NPI)
part_d_agg = part_d_clean.groupby('Prscrbr_NPI').agg({
    'Prscrbr_Last_Org_Name': 'first',
    'Prscrbr_First_Name': 'first',
    'Prscrbr_State_Abrvtn': 'first',
    'Prscrbr_Type': 'first',

    # How much care
    'Tot_Clms': 'sum',           # Total number of claims filed
    'Tot_30day_Fills': 'sum',    # Total monthly prescriptions written
    'Tot_Day_Suply': 'sum',      # Total days of medication supplied to patients

    # How much money
    'Tot_Drug_Cst': ['sum', 'mean'],  # sum = total cost; mean = avg cost per drug entry

    # Drug diversity and prescribing patterns
    'Brnd_Name': 'nunique',      # Number of unique brand-name drugs prescribed
    'Gnrc_Name': 'nunique',      # Number of unique generic drugs prescribed
}).reset_index()

# Flatten multi-level columns from aggregation
part_d_agg.columns = ['_'.join(col).strip('_') if isinstance(col, tuple) else col for col in part_d_agg.columns.values]

# Rename columns for clarity
column_mapping = {
    'Prscrbr_NPI': 'NPI',
    'Prscrbr_Last_Org_Name_first': 'Prescriber_Name_D',
    'Prscrbr_First_Name_first': 'Prescriber_FirstName_D',
    'Prscrbr_State_Abrvtn_first': 'Prescriber_State_D',
    'Prscrbr_Type_first': 'Prescriber_Type_D',
    'Tot_Clms_sum': 'Total_Claims_D',
    'Tot_30day_Fills_sum': 'Total_30Day_Fills_D',
    'Tot_Day_Suply_sum': 'Total_Day_Supply_D',
    'Tot_Drug_Cst_sum': 'Total_Drug_Cost_D',
    'Tot_Drug_Cst_mean': 'Avg_Drug_Cost_Per_Claim_D',
    'Brnd_Name_nunique': 'Unique_Brand_Drugs_D',
    'Gnrc_Name_nunique': 'Unique_Generic_Drugs_D',
}
part_d_agg.rename(columns=column_mapping, inplace=True)

print(f"✓ Part D aggregated to {len(part_d_agg):,} unique prescribers")
part_d_agg.head()


In [ ]:
# NPI overlap between Part B and Part D
npi_b = set(part_b_agg['NPI'])
npi_d = set(part_d_agg['NPI'])
npi_both = npi_b & npi_d

pct_b_in_d = len(npi_both) / len(npi_b) * 100  # % of Part B providers who also appear in Part D
pct_d_in_b = len(npi_both) / len(npi_d) * 100  # % of Part D providers who also appear in Part B

print("NPI Overlap between Part B and Part D:")
print(f"  Unique NPIs in Part B           : {len(npi_b):>10,}")
print(f"  Unique NPIs in Part D           : {len(npi_d):>10,}")
print(f"  NPIs present in both            : {len(npi_both):>10,}")
print(f"  NPIs only in Part B             : {len(npi_b - npi_d):>10,}")
print(f"  NPIs only in Part D             : {len(npi_d - npi_b):>10,}")
print(f"\n  % of Part B providers in Part D : {pct_b_in_d:.2f}%")
print(f"  % of Part D providers in Part B : {pct_d_in_b:.2f}%")


### 6.3 Create Fraud Labels from OIG Data

In [ ]:
print("\nCreating fraud labels from OIG exclusion list...")

oig_npi_list = oig_clean['NPI'].dropna().astype(str).str.strip()
# Filter out non-numeric NPIs
oig_npi_list = oig_npi_list[oig_npi_list.str.isnumeric()]
oig_npi_set = set(oig_npi_list.astype(int))
print(f"Total excluded providers in OIG: {len(oig_npi_set):,}")

# Merge Part B and Part D aggregations
print("\nMerging Part B and Part D data...")
providers_df = part_b_agg.merge(part_d_agg, on='NPI', how='outer', indicator=True)
print(f"Combined dataset: {len(providers_df):,} providers")
print(f"  - Only in Part B: {(providers_df['_merge'] == 'left_only').sum():,}")
print(f"  - Only in Part D: {(providers_df['_merge'] == 'right_only').sum():,}")
print(f"  - In both datasets: {(providers_df['_merge'] == 'both').sum():,}")

# Store merge indicator for analysis
providers_df['Data_Source'] = providers_df['_merge'].map({
    'left_only': 'Part_B_Only',
    'right_only': 'Part_D_Only',
    'both': 'Both'
})
providers_df.drop('_merge', axis=1, inplace=True)

# Consolidate provider information (prefer Part B, fallback to Part D)
print("\nConsolidating provider information...")
providers_df['Provider_Name'] = providers_df['Provider_Name'].fillna(providers_df['Prescriber_Name_D'])
providers_df['State'] = providers_df['State'].fillna(providers_df['Prescriber_State_D'])
providers_df['Provider_Type'] = providers_df['Provider_Type'].fillna(providers_df['Prescriber_Type_D'])

# Create fraud label by checking if NPI is in OIG exclusion list
print("\nCreating fraud labels...")
providers_df['Is_Fraudulent'] = providers_df['NPI'].isin(oig_npi_set).astype(int)

print(f"\nFraud Label Distribution:")
fraud_counts = providers_df['Is_Fraudulent'].value_counts()
print(f"  Non-Fraudulent (0): {fraud_counts.get(0, 0):,}")
print(f"  Fraudulent (1):     {fraud_counts.get(1, 0):,}")
print(f"  Fraud rate:         {providers_df['Is_Fraudulent'].mean()*100:.4f}%")

# Cross-tabulation: fraud by data source
print(f"\nFraud distribution by data source:")
fraud_by_source = pd.crosstab(providers_df['Data_Source'], providers_df['Is_Fraudulent'], margins=True)
print(fraud_by_source)

print(f"✓ Final merged dataset: {providers_df.shape[0]:,} providers × {providers_df.shape[1]} features")
providers_df.head()

In [ ]:
# Analyze missing values in the merged dataset
print("\n" + "="*80)
print("MISSING VALUE ANALYSIS - MERGED DATASET")
print("="*80)

# Calculate missing values
missing_data = pd.DataFrame({
    'Column': providers_df.columns,
    'Missing_Count': providers_df.isnull().sum().values,
    'Missing_Percent': (providers_df.isnull().sum().values / len(providers_df) * 100)
})
missing_data = missing_data.sort_values('Missing_Percent', ascending=False)

# Filter columns with missing values
missing_data_filtered = missing_data[missing_data['Missing_Count'] > 0]

print(f"\nTotal columns: {len(providers_df.columns)}")
print(f"Columns with missing values: {len(missing_data_filtered)}")
print(f"Columns without missing values: {len(providers_df.columns) - len(missing_data_filtered)}")

# Display table
print(f"\n{'Column':<40} {'Missing Count':<15} {'Missing %':<15}")
print("-" * 70)
for _, row in missing_data_filtered.iterrows():
    print(f"{row['Column']:<40} {int(row['Missing_Count']):<15,} {row['Missing_Percent']:<15.2f}")

# Create visualization
import plotly.graph_objects as go

# Only plot columns with missing values
plot_data = missing_data_filtered.head(20)  # Show top 20 if many columns

fig = go.Figure()

fig.add_trace(go.Bar(
    x=plot_data['Missing_Percent'],
    y=plot_data['Column'],
    orientation='h',
    marker=dict(
        color=plot_data['Missing_Percent'],
        colorscale='Reds',
        showscale=True,
        colorbar=dict(title="Missing %")
    ),
    text=[f"{val:.2f}%" for val in plot_data['Missing_Percent']],
    textposition='auto',
))

fig.update_layout(
    title="Missing Value Percentage by Column (Merged Dataset)",
    xaxis_title="Missing Percentage (%)",
    yaxis_title="Column Name",
    height=max(400, len(plot_data) * 25),
    showlegend=False,
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()

print("="*80)

### 6.3.1 Missing Value Analysis

### 6.4 Post-Merge Data Quality & Missing Value Handling


   - **Entity_Code**: Inferred from Provider_FirstName (397,548 values)
     - If FirstName exists → 'I' (Individual)
     - If FirstName empty/null → 'O' (Organization)
   - **Provider_FirstName**: Empty string for organizations (397,548 values)
   - **Zipcode**: State-based mode imputation (397,548 values)
   - **Part B metrics**: 0 for Part D-only providers (397,548 × 15 features)
     - Zero means "no Part B activity", not actual zero services
   - **Part D metrics**: 0 for Part B-only providers (468,667 × 13 features)
     - Zero means "no Part D activity", not actual zero prescriptions
   - **Prescriber info include Prescriber_Name_D, Prescriber_State_D, Prescriber_Type_D**: Copied from Provider info for Part B-only providers (468,667 × 3 fields)



### 6.4.1

In [ ]:
print("VERIFYING POST-MERGE IMPUTATION")

print(f"\nBefore imputation:")
print(f"  Missing values: {providers_df.isnull().sum().sum():,}")

print(f"\nAfter imputation:")
print(f"  Missing values: {providers_df.isnull().sum().sum():,}")

print(f"\nKey columns after imputation:")
print(f"  Entity_Code missing: {providers_df['Entity_Code'].isnull().sum()}")
print(f"  Provider_FirstName missing: {providers_df['Provider_FirstName'].isnull().sum()}")
print(f"  Zipcode missing: {providers_df['Zipcode'].isnull().sum()}")
print(f"  Prescriber_Name_D missing: {providers_df['Prescriber_Name_D'].isnull().sum()}")
print(f"  Prescriber_State_D missing: {providers_df['Prescriber_State_D'].isnull().sum()}")
print(f"  Prescriber_Type_D missing: {providers_df['Prescriber_Type_D'].isnull().sum()}")

print("SECTION 6.4 COMPLETE - FINAL DATASET SUMMARY")

### 6.4.2

In [ ]:
# Missing value imputation based on semantic logic
print("SEMANTIC MISSING VALUE IMPUTATION")

def impute_post_merge_missing(df):

    df_imputed = df.copy()
    imputation_log = []

    # Identify Part B-only and Part D-only providers.
    # The key principle applied throughout:
    #   - Part B-only features (Entity_Code, Provider_FirstName, Zipcode, RUCA_Code):
    #     Part D-only providers never had this data collected → impute 'Unknown'
    #   - Part D-only features (Prescriber_*):
    #     Part B-only providers never had this data collected → cross-fill if an
    #     equivalent Part B column exists, otherwise impute 'Unknown'
    #   - Cross-dataset features (State, Provider_Type):
    #     Already consolidated in Section 6.3 via fillna from the other dataset.
    #     If still missing after that cross-fill, no source has the data → 'Unknown'
    #     (using mode would fabricate a value with no real justification)
    part_d_only = df_imputed.get('Data_Source', pd.Series('', index=df_imputed.index)) == 'Part_D_Only'
    part_b_only = df_imputed.get('Data_Source', pd.Series('', index=df_imputed.index)) == 'Part_B_Only'

    # --- Entity_Code (Part B-only feature) ---
    if 'Entity_Code' in df_imputed.columns and 'Provider_FirstName' in df_imputed.columns:
        missing_entity = df_imputed['Entity_Code'].isnull()
        if missing_entity.sum() > 0:
            # Part D-only providers: not collected in Part D → 'Unknown'
            df_imputed.loc[missing_entity & part_d_only, 'Entity_Code'] = 'Unknown'
            # Part B providers still missing: infer from Provider_FirstName (I=Individual, O=Organization)
            has_firstname = df_imputed['Provider_FirstName'].notna() & (df_imputed['Provider_FirstName'] != '')
            df_imputed.loc[missing_entity & ~part_d_only & has_firstname, 'Entity_Code'] = 'I'
            df_imputed.loc[missing_entity & ~part_d_only & ~has_firstname, 'Entity_Code'] = 'O'
            imputation_log.append(f"Entity_Code: {missing_entity.sum()} values imputed "
                                  f"('Unknown' for Part D-only; I/O heuristic for Part B providers)")

    # --- Provider_FirstName (Part B-only feature) ---
    # Part D-only providers: not collected in Part D → 'Unknown'
    # Part B providers still missing: likely an organization (no first name) → ''
    if 'Provider_FirstName' in df_imputed.columns:
        missing_fn = df_imputed['Provider_FirstName'].isnull()
        if missing_fn.sum() > 0:
            df_imputed.loc[missing_fn & part_d_only, 'Provider_FirstName'] = 'Unknown'
            df_imputed.loc[missing_fn & ~part_d_only, 'Provider_FirstName'] = ''
            imputation_log.append(f"Provider_FirstName: {missing_fn.sum()} values imputed "
                                  f"('Unknown' for Part D-only; '' for Part B organizations)")

    # --- State (cross-dataset: already consolidated in Section 6.3) ---
    # If still missing here, neither dataset had a state for this provider → 'Unknown'
    for state_col in ['State', 'State_B', 'State_D']:
        if state_col in df_imputed.columns:
            missing_state = df_imputed[state_col].isnull()
            if missing_state.sum() > 0:
                df_imputed.loc[missing_state, state_col] = 'Unknown'
                imputation_log.append(f"{state_col}: {missing_state.sum()} values imputed to 'Unknown' "
                                      f"(no state data in either dataset)")

    # --- Provider_Type (cross-dataset: already consolidated in Section 6.3) ---
    # If still missing here, neither dataset had a specialty for this provider → 'Unknown'
    for type_col in ['Provider_Type', 'Provider_Type_B', 'Provider_Type_D']:
        if type_col in df_imputed.columns:
            missing_type = df_imputed[type_col].isnull()
            if missing_type.sum() > 0:
                df_imputed.loc[missing_type, type_col] = 'Unknown'
                imputation_log.append(f"{type_col}: {missing_type.sum()} values imputed to 'Unknown' "
                                      f"(no specialty data in either dataset)")

    # --- Zipcode (Part B-only feature) ---
    # Part D-only providers: not collected in Part D → 'Unknown'
    # Part B providers still missing: state-based mode, or '00000'
    if 'Zipcode' in df_imputed.columns:
        missing_zip = df_imputed['Zipcode'].isnull()
        if missing_zip.sum() > 0:
            df_imputed.loc[missing_zip & part_d_only, 'Zipcode'] = 'Unknown'
            missing_zip_b = df_imputed['Zipcode'].isnull()  # remaining Part B providers
            if missing_zip_b.sum() > 0:
                if 'State' in df_imputed.columns:
                    for state in df_imputed.loc[missing_zip_b, 'State'].unique():
                        state_mask = missing_zip_b & (df_imputed['State'] == state)
                        mode_zip = df_imputed[df_imputed['State'] == state]['Zipcode'].mode()
                        fill_val = mode_zip[0] if len(mode_zip) > 0 else '00000'
                        df_imputed.loc[state_mask, 'Zipcode'] = fill_val
                else:
                    df_imputed.loc[missing_zip_b, 'Zipcode'] = '00000'
            imputation_log.append(f"Zipcode: {missing_zip.sum()} values imputed "
                                  f"('Unknown' for Part D-only; state-based mode for Part B providers)")

    # --- Numeric features ---
    # Impute 0 to represent absence of activity for the missing dataset.
    # Note: RUCA_Code is a Part B-only feature — 0 is not a valid RUCA code, but serves
    # as a numeric sentinel for Part D-only providers who have no rurality data.
    numeric_cols = df_imputed.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if col not in ['NPI', 'Fraud_Flag']:
            missing_num = df_imputed[col].isnull()
            if missing_num.sum() > 0:
                df_imputed.loc[missing_num, col] = 0
                imputation_log.append(f"{col}: {missing_num.sum()} values imputed to 0 "
                                      f"(no activity / not in source dataset)")

    # --- Prescriber_* columns (Part D-only features) ---
    # For Part B-only providers: cross-fill from the equivalent Part B column where available,
    # then 'Unknown' for any remaining missing (no equivalent data collected).
    prescriber_crossfill = {
        'Prescriber_Name_D':      'Provider_Name',
        'Prescriber_FirstName_D': 'Provider_FirstName',
        'Prescriber_State_D':     'State',
        'Prescriber_Type_D':      'Provider_Type',
    }
    for d_col, b_col in prescriber_crossfill.items():
        if d_col in df_imputed.columns:
            missing_val = df_imputed[d_col].isnull()
            if missing_val.sum() > 0:
                if b_col in df_imputed.columns:
                    df_imputed.loc[missing_val, d_col] = df_imputed.loc[missing_val, b_col]
                still_missing = df_imputed[d_col].isnull()
                if still_missing.sum() > 0:
                    df_imputed.loc[still_missing, d_col] = 'Unknown'
                imputation_log.append(f"{d_col}: {missing_val.sum()} values imputed "
                                      f"(cross-filled from {b_col}; 'Unknown' for any remainder)")

    # --- Data_Source ---
    if 'Data_Source' in df_imputed.columns:
        missing_val = df_imputed['Data_Source'].isnull()
        if missing_val.sum() > 0:
            df_imputed.loc[missing_val, 'Data_Source'] = 'Unknown'
            imputation_log.append(f"Data_Source: {missing_val.sum()} values imputed to 'Unknown'")

    print("\nImputation Summary:")
    for log_entry in imputation_log:
        print(f"  • {log_entry}")
    if not imputation_log:
        print("  ✓ No missing values required imputation")

    return df_imputed

# Apply imputation
providers_df_clean = impute_post_merge_missing(providers_df)

# Verify no missing values remain
final_missing = providers_df_clean.isnull().sum().sum()
print(f"\n{'='*80}")
print(f"FINAL VERIFICATION: {final_missing} missing values remaining")
print(f"{'='*80}")
if final_missing == 0:
    print("✓ All missing values successfully imputed!")
else:
    print(f"⚠️  Still have {final_missing} missing values")
    remaining_missing = providers_df_clean.isnull().sum()
    print("\nRemaining missing values:")
    print(remaining_missing[remaining_missing > 0])


### 6.4.3

In [ ]:
providers_df_clean.head(10)

### 6.4.4

In [ ]:
print(f"\nBefore removal: {providers_df_clean.shape}")
print(f"Columns: {len(providers_df_clean.columns)}")

redundant_cols = ['Prescriber_Name_D', 'Prescriber_FirstName_D', 'Prescriber_State_D', 'Prescriber_Type_D']


print(f"\nRemoving Part D prescriber identity columns (redundant after consolidation in Section 6.3):")
for col in redundant_cols:
    if col in providers_df_clean.columns:
        print(f"  ✗ {col}")

providers_df_clean = providers_df_clean.drop(columns=redundant_cols, errors='ignore')


providers_df = providers_df_clean.copy()

print(f"\nAfter removal: {providers_df.shape}")
print(f"Columns: {len(providers_df.columns)}")
print(f"\nRemaining columns:")
print(providers_df.columns.tolist())

print("\n✓ Redundant columns removed successfully!")

In [ ]:
providers_df.head(10)

## 7. Exploratory Data Analysis (EDA)

### 7.1 Feature Engineering

In [ ]:

# ═══════════════════════════════════════════════════════════════════════
# Section 7.1  Advanced Feature Engineering
# ═══════════════════════════════════════════════════════════════════════

_n_cols_before = providers_df.shape[1]

print("=" * 60)
print("Section 7.1  Advanced Feature Engineering")
print("=" * 60)
print(f"\nStarting shape  : {providers_df.shape[0]:,} rows × {providers_df.shape[1]:,} columns")
print(f"Missing values  : {providers_df.isnull().sum().sum():,}")

# ─────────────────────────────────────────────────────────────────────────────
# GROUP 1 — Payment & Charge Ratio Features
# ─────────────────────────────────────────────────────────────────────────────
print("\n[1] Payment & charge ratio features ...")

providers_df['Charge_to_Allowed_Ratio'] = np.where(
    providers_df['Avg_Medicare_Allowed_B'] > 0,
    providers_df['Avg_Submitted_Charge_B'] / providers_df['Avg_Medicare_Allowed_B'],
    0
)

# Payment_to_Allowed_Ratio: Medicare payment ÷ allowed amount (should be ≤ 1).
# Very low values may indicate systematic write-offs or deductible gaming.
providers_df['Payment_to_Allowed_Ratio'] = np.where(
    providers_df['Avg_Medicare_Allowed_B'] > 0,
    providers_df['Avg_Medicare_Payment_B'] / providers_df['Avg_Medicare_Allowed_B'],
    0
)

# MaxCharge_Ratio: maximum vs. average submitted charge per provider.
# ✅ NEW — a high ratio means the provider occasionally files extreme charges
# far above their own average (hallmark of targeted upcoding on lucrative codes).
providers_df['MaxCharge_Ratio'] = np.where(
    providers_df['Avg_Submitted_Charge_B'] > 0,
    providers_df['Max_Submitted_Charge_B'] / providers_df['Avg_Submitted_Charge_B'],
    0
)

# Allowed_Charge_Gap: absolute dollar gap between submitted and allowed.
# ✅ NEW — captures raw over-billing magnitude (not just the ratio), useful
# for high-charge, low-volume fraudsters who game single expensive codes.
providers_df['Allowed_Charge_Gap'] = (
    providers_df['Avg_Submitted_Charge_B'] - providers_df['Avg_Medicare_Allowed_B']
)

# Payment_Efficiency: fraction of submitted charge that Medicare pays.
# ✅ NEW — consistently high (provider "knows" the exact allowed amount)
# or consistently low (claims denied en masse) both flag anomalies.
providers_df['Payment_Efficiency'] = np.where(
    providers_df['Avg_Submitted_Charge_B'] > 0,
    providers_df['Avg_Medicare_Payment_B'] / providers_df['Avg_Submitted_Charge_B'],
    0
)

# Standardized_Payment_Ratio: actual payment vs. geography-adjusted baseline.
# ✅ NEW — persistent premium above standardized amounts flags providers in
# low-cost areas who still receive high absolute payments (geographic fraud).
providers_df['Standardized_Payment_Ratio'] = np.where(
    providers_df['Avg_Medicare_Standardized_B'] > 0,
    providers_df['Avg_Medicare_Payment_B'] / providers_df['Avg_Medicare_Standardized_B'],
    0
)

# ─────────────────────────────────────────────────────────────────────────────
# GROUP 2 — Service Intensity & Volume Features (Part B)
# Rationale: fraudulent providers often inflate service counts, beneficiary-day
# services, or concentrate billing on a small number of high-paying codes.
# ─────────────────────────────────────────────────────────────────────────────
print("[2] Service intensity & volume features ...")

# Services_per_Beneficiary_B: average services billed per patient.
# Fraud providers inflate this via phantom services or medically unnecessary procedures.
providers_df['Services_per_Beneficiary_B'] = np.where(
    providers_df['Total_Beneficiaries_B'] > 0,
    providers_df['Total_Services_B'] / providers_df['Total_Beneficiaries_B'],
    0
)

# BeneDay_Per_Beneficiary_B: beneficiary-day services per unique patient.
# ✅ NEW — inflated beneficiary-day counts (e.g., billing 20 service-days per
# patient) are a specific red flag for home-health and nursing-facility fraud.
providers_df['BeneDay_Per_Beneficiary_B'] = np.where(
    providers_df['Total_Beneficiaries_B'] > 0,
    providers_df['Total_BeneDay_Services_B'] / providers_df['Total_Beneficiaries_B'],
    0
)

# Services_Per_HCPCS: average services per unique procedure code.
# ✅ NEW — high concentration means repeatedly billing the same few codes
# at very high volume (classic upcoding / unbundling pattern).
providers_df['Services_Per_HCPCS'] = np.where(
    providers_df['Unique_HCPCS_B'] > 0,
    providers_df['Total_Services_B'] / providers_df['Unique_HCPCS_B'],
    0
)

# HCPCS_Diversity: unique HCPCS codes as a share of total services (0–1).
# ✅ NEW — the inverse of Services_Per_HCPCS; values near 0 = concentrated
# billing. Distinct from Services_Per_HCPCS in scale; useful for linear models.
providers_df['HCPCS_Diversity'] = np.where(
    providers_df['Total_Services_B'] > 0,
    providers_df['Unique_HCPCS_B'] / providers_df['Total_Services_B'],
    0
)

# Procedure_Breadth: named alias for Unique_HCPCS_B.
# ✅ NEW — making the concept explicit aids feature-importance interpretation
# (breadth of procedure coding as a standalone concept).
providers_df['Procedure_Breadth'] = providers_df['Unique_HCPCS_B']

# Total_Patient_Volume: clean alias for Total_Beneficiaries_B.
providers_df['Total_Patient_Volume'] = providers_df['Total_Beneficiaries_B']

# Binary participation indicators
providers_df['Has_PartB'] = (providers_df['Total_Services_B'] > 0).astype(int)
providers_df['Has_PartD'] = (providers_df['Total_Claims_D']   > 0).astype(int)
providers_df['Has_Both']  = (
    (providers_df['Has_PartB'] == 1) & (providers_df['Has_PartD'] == 1)
).astype(int)

# ─────────────────────────────────────────────────────────────────────────────
# GROUP 3 — Drug Cost & Prescribing Pattern Features (Part D)
# Rationale: drug fraud manifests as inflated cost-per-claim, excessive brand-
# name prescribing, abnormally short supply durations, or high daily drug costs.
# ─────────────────────────────────────────────────────────────────────────────
print("[3] Drug cost & prescribing pattern features ...")

# Drug_Cost_Per_Claim_D: proper weighted average = Total_Drug_Cost_D / Total_Claims_D.
# ✅ NEW — more accurate than Avg_Drug_Cost_Per_Claim_D (which is mean-of-means
# across drug entries and may differ for providers with diverse drug portfolios).
providers_df['Drug_Cost_Per_Claim_D'] = np.where(
    providers_df['Total_Claims_D'] > 0,
    providers_df['Total_Drug_Cost_D'] / providers_df['Total_Claims_D'],
    0
)

# Avg_Cost_per_Claim_D: legacy alias (keeps backward compatibility with later cells).
providers_df['Avg_Cost_per_Claim_D'] = providers_df['Drug_Cost_Per_Claim_D']

# Drug_Cost_Per_Day: total drug spend ÷ total days of supply.
# ✅ NEW — captures the per-day cost of treatment. Persistently high values
# suggest billing for expensive/high-dose medications disproportionate to practice
# volume (key signal for specialty-drug and opioid fraud).
providers_df['Drug_Cost_Per_Day'] = np.where(
    providers_df['Total_Day_Supply_D'] > 0,
    providers_df['Total_Drug_Cost_D'] / providers_df['Total_Day_Supply_D'],
    0
)

# Fills_Per_Claim_D: 30-day fills per filed claim.
# ✅ NEW — values > 1 mean multiple fills per claim script; very high values
# may indicate drug mills generating many short-fill prescriptions per encounter.
providers_df['Fills_Per_Claim_D'] = np.where(
    providers_df['Total_Claims_D'] > 0,
    providers_df['Total_30Day_Fills_D'] / providers_df['Total_Claims_D'],
    0
)

# Days_Per_Claim_D: average days supply per claim.
# ✅ NEW — extremely short supply (e.g., 3-day fills) drives up claim counts
# without medical justification; a classic sign of pill-mill / drug diversion.
providers_df['Days_Per_Claim_D'] = np.where(
    providers_df['Total_Claims_D'] > 0,
    providers_df['Total_Day_Supply_D'] / providers_df['Total_Claims_D'],
    0
)

# ─────────────────────────────────────────────────────────────────────────────
# GROUP 4 — Drug Diversity & Brand Features
# ─────────────────────────────────────────────────────────────────────────────
print("[4] Drug diversity & brand features ...")

# Brand_Ratio: share of unique drug names that are brand-name (0–1).
# ✅ NEW — brand drugs are significantly more expensive than generics;
# a high Brand_Ratio combined with high Drug_Cost_Per_Claim signals aggressive
# brand prescribing that inflates Medicare drug spend (specialty-drug fraud).
providers_df['Brand_Ratio'] = np.where(
    (providers_df['Unique_Brand_Drugs_D'] + providers_df['Unique_Generic_Drugs_D']) > 0,
    providers_df['Unique_Brand_Drugs_D'] /
        (providers_df['Unique_Brand_Drugs_D'] + providers_df['Unique_Generic_Drugs_D']),
    0
)

# Total_Service_Diversity: breadth across both Part B and Part D.
providers_df['Total_Service_Diversity'] = (
    providers_df['Service_Type_Diversity_B'] +
    providers_df['Unique_Brand_Drugs_D'] +
    providers_df['Unique_Generic_Drugs_D']
)

# ─────────────────────────────────────────────────────────────────────────────
# GROUP 5 — Cross-Part (B ↔ D) Interaction Features
# Rationale: comparing Part B services to Part D claims surfaces specialty
# mismatches and unusual billing composition between care modalities.
# ─────────────────────────────────────────────────────────────────────────────
print("[5] Cross-part B ↔ D features ...")

# Service_to_Drug_Ratio: Part B services relative to Part D claims.
# ✅ NEW — an unusually high ratio for a prescribing-heavy specialty (e.g., a
# psychiatrist with 1,000 Part B services but only 2 Part D claims) flags a
# billing pattern inconsistent with specialty norms.
providers_df['Service_to_Drug_Ratio'] = np.where(
    providers_df['Total_Claims_D'] > 0,
    providers_df['Total_Services_B'] / providers_df['Total_Claims_D'],
    0
)

# ─────────────────────────────────────────────────────────────────────────────
# GROUP 6 — Log-Transformed Volume Features
# Rationale: all raw volume/cost features are heavily right-skewed.  log(1 + x)
# compresses extreme outliers without losing them, improving linear-model and
# distance-based model performance significantly.
# ─────────────────────────────────────────────────────────────────────────────
print("[6] Log-transform features (log1p, zero-safe) ...")

providers_df['log_Total_Services_B']      = np.log1p(providers_df['Total_Services_B'])
providers_df['log_Total_Beneficiaries_B'] = np.log1p(providers_df['Total_Beneficiaries_B'])
providers_df['log_Total_Claims_D']        = np.log1p(providers_df['Total_Claims_D'])
providers_df['log_Total_Drug_Cost_D']     = np.log1p(providers_df['Total_Drug_Cost_D'])
providers_df['log_Total_Day_Supply_D']    = np.log1p(providers_df['Total_Day_Supply_D'])

# ─────────────────────────────────────────────────────────────────────────────
# GROUP 7 — Within-Provider-Type Z-scores
# Rationale: this is the MOST IMPORTANT contextual normalisation.  A neurologist
# billing 500 services/patient is extreme; for a clinical lab it is routine.
# Z-scoring within Provider_Type makes anomalies peer-relative — exactly how
# fraud investigators approach outlier detection.
# ─────────────────────────────────────────────────────────────────────────────
print("[7] Within-Provider_Type z-score features ...")

for raw_col, z_col in [
    ('Total_Services_B',      'Z_Total_Services_B'),
    ('Total_Drug_Cost_D',     'Z_Total_Drug_Cost_D'),
    ('Avg_Medicare_Payment_B','Z_Avg_Medicare_Payment_B'),
]:
    _grp  = providers_df.groupby('Provider_Type')[raw_col]
    _mean = _grp.transform('mean')
    _std  = _grp.transform('std').replace(0, np.nan)
    providers_df[z_col] = ((providers_df[raw_col] - _mean) / _std).fillna(0)

# ─────────────────────────────────────────────────────────────────────────────
# GROUP 8 — Binary High-Value Anomaly Flags
# Rationale: explicit binary thresholds for the ≥ 95th percentile within peer
# group (z > 1.645).  Tree-based models already discover splits automatically,
# but these flags make the concept explicit and aid model interpretability.
# ─────────────────────────────────────────────────────────────────────────────
print("[8] Binary high-value anomaly flags (z > 1.645 = top-5% within specialty) ...")

providers_df['High_Service_Volume_Flag'] = (
    providers_df['Z_Total_Services_B']       > 1.645).astype(int)
providers_df['High_Drug_Cost_Flag']      = (
    providers_df['Z_Total_Drug_Cost_D']      > 1.645).astype(int)
providers_df['High_Payment_Flag']        = (
    providers_df['Z_Avg_Medicare_Payment_B'] > 1.645).astype(int)

# ─────────────────────────────────────────────────────────────────────────────
# GROUP 9 — Interaction / Composite Features
# Rationale: multiplicative interactions encode compound risk that neither
# factor alone can express for linear models.
# ─────────────────────────────────────────────────────────────────────────────
print("[9] Interaction features ...")

# High_Service_High_Payment: services-per-patient × average Medicare payment.
# ✅ NEW — captures the "dangerous combination": both high service intensity AND
# high payment per service. A provider who sees many patients AND charges a lot
# is the archetypal high-impact fraud pattern.
providers_df['High_Service_High_Payment'] = (
    providers_df['Services_per_Beneficiary_B'] * providers_df['Avg_Medicare_Payment_B']
)

# Drug_Intensity: cost-per-claim × brand ratio.
# ✅ NEW — a compound indicator encoding "expensive AND predominantly brand-name".
# Providers at both extremes simultaneously represent the highest-risk prescribers.
providers_df['Drug_Intensity'] = (
    providers_df['Drug_Cost_Per_Claim_D'] * providers_df['Brand_Ratio']
)

# Charge_x_Volume: average submitted charge × total services.
# ✅ NEW — absolute billing exposure magnitude. Helps tree models split on total
# dollar impact rather than just rate or count alone.
providers_df['Charge_x_Volume'] = (
    providers_df['Avg_Submitted_Charge_B'] * providers_df['Total_Services_B']
)

# ─────────────────────────────────────────────────────────────────────────────
# Summary
# ─────────────────────────────────────────────────────────────────────────────
_n_new = providers_df.shape[1] - _n_cols_before
print(f"\n{'='*60}")
print(f"Final shape     : {providers_df.shape[0]:,} rows × {providers_df.shape[1]:,} columns")
print(f"Base columns    : {_n_cols_before}")
print(f"New features    : {_n_new}")
print(f"Missing values  : {providers_df.isnull().sum().sum():,}")
print("✓ Feature engineering complete!")

providers_df.head()


#### 7.1.1 Summary Statistics for Engineered Features

In [ ]:

# ── 7.1.1  Summary Statistics for All Engineered Features ─────────────────────
print("=" * 80)
print("7.1.1  Summary Statistics — Engineered Features")
print("=" * 80)

engineered_features = {
    # ── Group 1: Payment & Charge Ratios ──────────────────────────────────────
    "--- GROUP 1: Payment & Charge Ratios ---": [],
    "Charge_to_Allowed_Ratio"      : "Over-billing rate vs. Medicare allowed (>1 = upcoding risk)",
    "Payment_to_Allowed_Ratio"     : "Medicare payment as fraction of allowed (legitimately ≤1)",
    "MaxCharge_Ratio"              : "Max vs. avg charge — outlier charge spike detector",
    "Allowed_Charge_Gap"           : "Absolute $ over-billing per service line",
    "Payment_Efficiency"           : "Fraction of submitted charge actually paid by Medicare",
    "Standardized_Payment_Ratio"   : "Payment premium above geography-adjusted baseline",

    # ── Group 2: Service Intensity ─────────────────────────────────────────────
    "--- GROUP 2: Service Intensity (Part B) ---": [],
    "Services_per_Beneficiary_B"   : "Average services billed per unique patient",
    "BeneDay_Per_Beneficiary_B"    : "Beneficiary-day services per patient (phantom-billing flag)",
    "Services_Per_HCPCS"           : "Billing code concentration (repetitive-code flag)",
    "HCPCS_Diversity"              : "Code diversity ratio — low = concentrated billing",
    "Procedure_Breadth"            : "Unique HCPCS codes (named alias for Unique_HCPCS_B)",
    "Total_Patient_Volume"         : "Unique patient count (alias for Total_Beneficiaries_B)",
    "Has_PartB"                    : "Binary: provider has Part B activity",
    "Has_PartD"                    : "Binary: provider has Part D activity",
    "Has_Both"                     : "Binary: provider participates in both Part B and D",

    # ── Group 3: Drug Cost & Prescribing ──────────────────────────────────────
    "--- GROUP 3: Drug Cost & Prescribing (Part D) ---": [],
    "Drug_Cost_Per_Claim_D"        : "Weighted avg drug cost per filed claim (sum÷sum)",
    "Avg_Cost_per_Claim_D"         : "Alias for Drug_Cost_Per_Claim_D (backward compat.)",
    "Drug_Cost_Per_Day"            : "Daily drug cost — high = expensive/high-dose prescribing",
    "Fills_Per_Claim_D"            : "30-day fills per claim — >1 flags drug-mill patterns",
    "Days_Per_Claim_D"             : "Avg days supply per claim — very low = frequent refills",

    # ── Group 4: Drug Diversity & Brand ───────────────────────────────────────
    "--- GROUP 4: Drug Diversity & Brand ---": [],
    "Brand_Ratio"                  : "Fraction of unique drugs that are brand-name (0–1)",
    "Total_Service_Diversity"      : "Breadth of services across Part B + Part D",

    # ── Group 5: Cross-Part Features ──────────────────────────────────────────
    "--- GROUP 5: Cross-Part (B ↔ D) ---": [],
    "Service_to_Drug_Ratio"        : "Part B services / Part D claims (specialty mismatch flag)",

    # ── Group 6: Log Transforms ────────────────────────────────────────────────
    "--- GROUP 6: Log Transforms ---": [],
    "log_Total_Services_B"         : "log(1 + Total_Services_B) — right-skew compression",
    "log_Total_Beneficiaries_B"    : "log(1 + Total_Beneficiaries_B)",
    "log_Total_Claims_D"           : "log(1 + Total_Claims_D)",
    "log_Total_Drug_Cost_D"        : "log(1 + Total_Drug_Cost_D)",
    "log_Total_Day_Supply_D"       : "log(1 + Total_Day_Supply_D)",

    # ── Group 7: Within-Specialty Z-scores ────────────────────────────────────
    "--- GROUP 7: Within-Provider_Type Z-scores ---": [],
    "Z_Total_Services_B"           : "Peer-relative service volume (vs. same Provider_Type)",
    "Z_Total_Drug_Cost_D"          : "Peer-relative drug spend (vs. same Provider_Type)",
    "Z_Avg_Medicare_Payment_B"     : "Peer-relative payment per service (vs. same Provider_Type)",

    # ── Group 8: Binary Anomaly Flags ─────────────────────────────────────────
    "--- GROUP 8: Binary Anomaly Flags (top-5% within specialty) ---": [],
    "High_Service_Volume_Flag"     : "1 if Z_Total_Services_B > 1.645",
    "High_Drug_Cost_Flag"          : "1 if Z_Total_Drug_Cost_D > 1.645",
    "High_Payment_Flag"            : "1 if Z_Avg_Medicare_Payment_B > 1.645",

    # ── Group 9: Interaction Features ─────────────────────────────────────────
    "--- GROUP 9: Interaction Features ---": [],
    "High_Service_High_Payment"    : "Services_per_Beneficiary_B × Avg_Medicare_Payment_B",
    "Drug_Intensity"               : "Drug_Cost_Per_Claim_D × Brand_Ratio",
    "Charge_x_Volume"              : "Avg_Submitted_Charge_B × Total_Services_B (total exposure)",
}

# ── Print statistics for numeric features ────────────────────────────────────
summary_data = []
for key, desc in engineered_features.items():
    if key.startswith("---"):
        print(f"\n{key}")
        print("-" * 80)
        continue
    if key not in providers_df.columns:
        print(f"  ⚠ MISSING: {key}")
        continue
    col = providers_df[key]
    fraud_mean    = providers_df.loc[providers_df['Is_Fraudulent'] == 1, key].mean()
    nonfr_mean    = providers_df.loc[providers_df['Is_Fraudulent'] == 0, key].mean()
    fraud_lift    = (fraud_mean / nonfr_mean) if nonfr_mean != 0 else float('nan')
    print(f"  {key:<32}  min={col.min():>10.3f}  max={col.max():>12.3f}  "
          f"mean={col.mean():>10.3f}  fraud_lift={fraud_lift:>6.2f}x  | {desc[:40]}")
    summary_data.append({
        'Feature'          : key,
        'Min'              : col.min(),
        'Max'              : col.max(),
        'Mean'             : col.mean(),
        'Median'           : col.median(),
        'Fraud Mean'       : fraud_mean,
        'Non-Fraud Mean'   : nonfr_mean,
        'Fraud Lift'       : fraud_lift,
        'Description'      : desc,
    })

summary_df = pd.DataFrame(summary_data)

print(f"\n{'='*80}")
print(f"✓ Total engineered features tracked : {len(summary_data)}")
print(f"✓ Successfully created              : {sum(1 for d in summary_data if d['Feature'] in providers_df.columns)}")
print(f"\nTop features by Fraud Lift (fraud mean / non-fraud mean):")
top_lift = (summary_df[summary_df['Fraud Lift'].notna() & np.isfinite(summary_df['Fraud Lift'])]
            .nlargest(10, 'Fraud Lift')[['Feature', 'Fraud Mean', 'Non-Fraud Mean', 'Fraud Lift']])
print(top_lift.to_string(index=False))


### 7.2

In [ ]:

# Create fraud analysis by state
fraud_by_state = providers_df[providers_df['Is_Fraudulent'] == 1]['State'].value_counts().head(15)

fraud_rate_by_state = (
    providers_df.groupby('State')['Is_Fraudulent'].agg(['sum', 'count'])
)
fraud_rate_by_state['fraud_rate'] = (fraud_rate_by_state['sum'] / fraud_rate_by_state['count'] * 100)
fraud_rate_by_state = fraud_rate_by_state.sort_values('fraud_rate', ascending=False).head(15)

fig = make_subplots(
    rows=1, cols=1,
    subplot_titles=('Top 15 States by Fraud Rate %',)
)

fig.add_trace(
    go.Bar(x=fraud_rate_by_state['fraud_rate'], y=fraud_rate_by_state.index, 
           orientation='h', name='Fraud Rate %', marker_color='red'),
    row=1, col=1
)

fig.update_layout(height=400, showlegend=False, title_text="Geographic Distribution of Healthcare Fraud")
fig.update_xaxes(title_text="Fraud Rate (%)", row=1, col=1)
fig.update_yaxes(title_text="State", row=1, col=1)
fig.show()

print(f"\nTop states by fraud count:")
print(fraud_by_state)

print(f"\nTop states by fraud rate:")
print(fraud_rate_by_state[['sum', 'count', 'fraud_rate']].head(15))


### 7.3 Specialty Analysis

In [ ]:
# Fraud Analysis by Provider Type
print("Analyzing fraud patterns by provider specialty...")

fraud_rate_by_type = (
    providers_df.groupby('Provider_Type')['Is_Fraudulent'].agg(['sum', 'count'])
)
fraud_rate_by_type = fraud_rate_by_type[fraud_rate_by_type['count'] >= 100] 
fraud_rate_by_type['fraud_rate'] = (fraud_rate_by_type['sum'] / fraud_rate_by_type['count'] * 100)
fraud_rate_by_type = fraud_rate_by_type.sort_values('fraud_rate', ascending=False).head(15)

fig = go.Figure()

fig.add_trace(
    go.Bar(y=fraud_rate_by_type.index, x=fraud_rate_by_type['fraud_rate'], orientation='h',
           marker_color='darkred', text=fraud_rate_by_type['fraud_rate'].round(2),
           texttemplate='%{text}%', textposition='outside')
)

fig.update_layout(
    height=600, 
    title_text="Healthcare Fraud Rate by Provider Specialty (min 100 providers)",
    xaxis_title="Fraud Rate (%)",
    yaxis_title="Provider Type",
    showlegend=False
)

fig.show()

print(f"\nTop 15 specialties by fraud rate:")
print(fraud_rate_by_type[['sum', 'count', 'fraud_rate']])


### 7.4 Correlation Analysis

In [ ]:

print("Analyzing feature correlations...")

# Select only numeric features for correlation
numeric_features = providers_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_features.remove('NPI') 


corr_matrix = providers_df[numeric_features].corr()


plt.figure(figsize=(16, 14))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16, pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


fraud_correlations = corr_matrix['Is_Fraudulent'].sort_values(ascending=False)
print("\nTop 15 features correlated with fraud:")
print(fraud_correlations.head(16)[1:]) 

print("\nBottom 10 features (negative correlation with fraud):")
print(fraud_correlations.tail(10))

## 8. Statistical Feature Selection

We'll use statistical methods to identify and remove redundant features:
1. **Correlation Analysis**: Remove highly correlated features (multicollinearity)
2. **Variance Threshold**: Remove low-variance features
3. **Chi-Square Test**: Statistical significance for feature-target relationship

### 8.1

In [ ]:
from sklearn.feature_selection import VarianceThreshold

print("SECTION 8: STATISTICAL FEATURE SELECTION")


# Drop non-predictive columns (identifiers, labels, categorical descriptors)
columns_to_drop_for_selection = ['NPI', 'Is_Fraudulent', 'Provider_Name', 'Provider_FirstName', 
                                   'State', 'Zipcode', 'Provider_Type', 'Entity_Code', 'Data_Source']
existing_cols_to_drop_selection = [col for col in columns_to_drop_for_selection if col in providers_df.columns]

print(f"\n  • Dropping {len(existing_cols_to_drop_selection)} non-predictive columns: {existing_cols_to_drop_selection}")

X_all = providers_df.drop(columns=existing_cols_to_drop_selection, axis=1)
y_all = providers_df['Is_Fraudulent']

print(f"\n✓ Total predictive features: {X_all.shape[1]}")
print(f"✓ Total samples: {X_all.shape[0]:,}")
print(f"✓ Fraud cases: {y_all.sum():,} ({y_all.sum()/len(y_all)*100:.2f}%)")

# STEP 0: Train-Test Split (BEFORE feature selection to prevent data leakage)
print("\n" + "-"*80)
print("STEP 0: TRAIN-TEST SPLIT (80-20)")
print("-"*80)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_all
)

print(f"\nTrain set: {X_train.shape[0]:,} samples ({y_train.sum():,} fraud cases, {y_train.sum()/len(y_train)*100:.2f}%)")
print(f"Test set:  {X_test.shape[0]:,} samples ({y_test.sum():,} fraud cases, {y_test.sum()/len(y_test)*100:.2f}%)")

# Step 1: Remove highly correlated features (multicollinearity) - TRAIN DATA ONLY
print("\n" + "-"*80)
print("STEP 1: CORRELATION-BASED FEATURE REMOVAL (on training data)")
print("-"*80)

correlation_matrix = X_train.corr().abs()
high_corr_threshold = 0.90
upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

# Find features to drop
to_drop_corr = [column for column in upper_triangle.columns 
                if any(upper_triangle[column] > high_corr_threshold)]

print(f"\nCorrelation threshold: {high_corr_threshold}")
print(f"Highly correlated features to remove: {len(to_drop_corr)}")
if to_drop_corr:
    print("\nFeatures removed due to high correlation:")
    for feat in to_drop_corr:
        correlated_with = upper_triangle[feat][upper_triangle[feat] > high_corr_threshold].index.tolist()
        if correlated_with:
            print(f"  • {feat} (corr > {high_corr_threshold} with: {correlated_with[0]})")

# Apply to both train and test
X_train_reduced = X_train.drop(columns=to_drop_corr, errors='ignore')
X_test_reduced = X_test.drop(columns=to_drop_corr, errors='ignore')
print(f"\n✓ Features after correlation removal: {X_train_reduced.shape[1]}")

# Step 2: Variance Threshold - TRAIN DATA ONLY
print("\n" + "-"*80)
print("STEP 2: VARIANCE THRESHOLD (on training data)")
print("-"*80)

var_threshold = 0.01
selector_var = VarianceThreshold(threshold=var_threshold)
selector_var.fit(X_train_reduced)

feature_variances = X_train_reduced.var()
low_var_features = feature_variances[feature_variances < var_threshold].index.tolist()

print(f"\nVariance threshold: {var_threshold}")
print(f"Low variance features to remove: {len(low_var_features)}")
if low_var_features:
    print("\nLow variance features:")
    for feat in low_var_features:
        print(f"  • {feat} (variance: {feature_variances[feat]:.6f})")

# Apply transformation to both train and test
X_train_reduced = pd.DataFrame(
    selector_var.transform(X_train_reduced),
    columns=X_train_reduced.columns[selector_var.get_support()],
    index=X_train_reduced.index
)
X_test_reduced = pd.DataFrame(
    selector_var.transform(X_test_reduced),
    columns=X_test_reduced.columns[selector_var.get_support()],
    index=X_test_reduced.index
)
print(f"\n✓ Features after variance threshold: {X_train_reduced.shape[1]}")

# Step 3: Chi-Square Test - TRAIN DATA ONLY
print("\n" + "-"*80)
print("STEP 3: CHI-SQUARE TEST (on training data)")
print("-"*80)

non_negative_features = X_train_reduced.columns[(X_train_reduced >= 0).all()].tolist()

if len(non_negative_features) > 0:
    print(f"\nNon-negative features for chi-square test: {len(non_negative_features)}")
    
    chi_scores, p_values = chi2(X_train_reduced[non_negative_features], y_train)
    
    chi_scores_df = pd.DataFrame({
        'Feature': non_negative_features,
        'Chi2_Score': chi_scores,
        'P_Value': p_values
    }).sort_values('Chi2_Score', ascending=False)
    
    print("\nTop 10 features by Chi-Square score:")
    print(chi_scores_df.head(10).to_string(index=False))
    
    significant_features = chi_scores_df[chi_scores_df['P_Value'] < 0.05]['Feature'].tolist()
    print(f"\nStatistically significant features (p < 0.05): {len(significant_features)}")
else:
    print("\n⚠️ No non-negative features available for chi-square test")
    significant_features = []
    chi_scores_df = None

# Summary
print("\n" + "="*80)
print("FEATURE SELECTION SUMMARY")
print("="*80)

print(f"\n1. Original features: {X_all.shape[1]}")
print(f"2. After correlation removal: {X_all.shape[1] - len(to_drop_corr)}")
print(f"3. After variance threshold: {X_train_reduced.shape[1]}")
print(f"4. Statistically significant (chi-square): {len(significant_features)}")

print(f"\n✓ Final features: {X_train_reduced.shape[1]}")
print(f"✓ Features removed: {X_all.shape[1] - X_train_reduced.shape[1]}")
print(f"✓ Reduction: {(1 - X_train_reduced.shape[1]/X_all.shape[1])*100:.1f}%")

selected_features_statistical = X_train_reduced.columns.tolist()
print(f"\nSelected features list ({len(selected_features_statistical)} features):")
for i, feat in enumerate(selected_features_statistical, 1):
    print(f"  {i}. {feat}")
feature_selection_results = {
    'original_features': X_all.columns.tolist(),
    'selected_features': selected_features_statistical,
    'removed_by_correlation': to_drop_corr,
    'removed_by_variance': low_var_features,
    'chi_scores': chi_scores_df
}

print("\n✓ STATISTICAL FEATURE SELECTION COMPLETE")
print("✓ Feature selection applied on training data only - no data leakage!")

In [ ]:
# Export the current pre-modeling train/test splits from Section 8.1
export_dir = OUTPUT_DIR / 'datasets'
export_dir.mkdir(parents=True, exist_ok=True)

# Use the latest reduced sets created in Section 8.1
X_train_current = X_train_reduced.copy() if 'X_train_reduced' in globals() else X_train.copy()
X_test_current = X_test_reduced.copy() if 'X_test_reduced' in globals() else X_test.copy()
y_train_current = y_train.copy()
y_test_current = y_test.copy()

# Combine features + target for easier reload
train_export = X_train_current.copy()
train_export['Is_Fraudulent'] = y_train_current.values

test_export = X_test_current.copy()
test_export['Is_Fraudulent'] = y_test_current.values

# Save separate files
X_train_current.to_csv(export_dir / 'X_train_reduced.csv', index=True, index_label='original_index')
X_test_current.to_csv(export_dir / 'X_test_reduced.csv', index=True, index_label='original_index')
y_train_current.to_csv(export_dir / 'y_train.csv', index=True, index_label='original_index', header=True)
y_test_current.to_csv(export_dir / 'y_test.csv', index=True, index_label='original_index', header=True)

# Save combined files
train_export.to_csv(export_dir / 'train_set_reduced.csv', index=True, index_label='original_index')
test_export.to_csv(export_dir / 'test_set_reduced.csv', index=True, index_label='original_index')

# Save feature list
pd.Series(X_train_current.columns, name='feature').to_csv(
    export_dir / 'selected_features_statistical.csv', index=False
)

print("✓ Train/test sets exported")
print(f"Export folder: {export_dir}")
print(f"Train features: {X_train_current.shape}")
print(f"Test features:  {X_test_current.shape}")
print(f"Train labels:   {y_train_current.shape}")
print(f"Test labels:    {y_test_current.shape}")
print("\nFiles created:")
for file_name in [
    'X_train_reduced.csv',
    'X_test_reduced.csv',
    'y_train.csv',
    'y_test.csv',
    'train_set_reduced.csv',
    'test_set_reduced.csv',
    'selected_features_statistical.csv'
]:
    print(f"  - {file_name}")

In [ ]:
# Reload the exported train/test datasets from Section 8.1
export_dir = OUTPUT_DIR / 'datasets'

X_train_reduced = pd.read_csv(export_dir / 'X_train_reduced.csv', index_col='original_index')
X_test_reduced = pd.read_csv(export_dir / 'X_test_reduced.csv', index_col='original_index')

y_train = pd.read_csv(export_dir / 'y_train.csv', index_col='original_index').iloc[:, 0]
y_test = pd.read_csv(export_dir / 'y_test.csv', index_col='original_index').iloc[:, 0]

selected_features_statistical = pd.read_csv(
    export_dir / 'selected_features_statistical.csv'
)['feature'].tolist()

# Optional: reload combined files too
train_export = pd.read_csv(export_dir / 'train_set_reduced.csv', index_col='original_index')
test_export = pd.read_csv(export_dir / 'test_set_reduced.csv', index_col='original_index')

print("✓ Dumped files loaded successfully")
print(f"Folder: {export_dir}")
print(f"X_train_reduced: {X_train_reduced.shape}")
print(f"X_test_reduced:  {X_test_reduced.shape}")
print(f"y_train:         {y_train.shape}")
print(f"y_test:          {y_test.shape}")
print(f"Selected features: {len(selected_features_statistical)}")

# Quick validation
print("\nFirst 5 selected features:")
print(selected_features_statistical[:5])

## 9. Data Scaling & Class Balancing

### 9.1 Data Summary (Using Pre-Split Data from Section 8.1)

In [ ]:

print("="*80)
print("DATA PREPARATION SUMMARY")
print("="*80)

print("\n📊 Using pre-split data from Section 8.1:")
print(f"   • Train-test split already performed (80-20 stratified)")
print(f"   • Feature selection applied on training data only")
print(f"   • No data leakage - test data never influenced feature selection")

print(f"\n✓ Training set shape: {X_train_reduced.shape}")
print(f"✓ Test set shape: {X_test_reduced.shape}")

print(f"\n✓ Training fraud rate: {y_train.mean()*100:.4f}% ({y_train.sum():,} / {len(y_train):,})")
print(f"✓ Test fraud rate: {y_test.mean()*100:.4f}% ({y_test.sum():,} / {len(y_test):,})")

print(f"\n✓ Selected features for modeling: {len(selected_features_statistical)}")
print("\nFeature list:")
for i, feat in enumerate(selected_features_statistical, 1):
    print(f"  {i:2d}. {feat}")

# Use the reduced datasets for modeling
X_train = X_train_reduced.copy()
X_test = X_test_reduced.copy()
feature_names = X_train.columns.tolist()

print(f"\n✓ Ready for modeling with {len(feature_names)} features")
print("="*80)

### 9.2 Handle Class Imbalance

In [ ]:
# Handle class imbalance with SMOTE-ENN (two-step)


from imblearn.over_sampling import SMOTE as _SMOTE
from imblearn.under_sampling import EditedNearestNeighbours as _ENN

print("Handling class imbalance with SMOTE-ENN (two-step)...")

n_fraud    = int((y_train == 1).sum())
n_nonfraud = int((y_train == 0).sum())

print(f"\nBefore resampling:")
print(f"  Class 0 (Non-Fraud): {n_nonfraud:,}")
print(f"  Class 1 (Fraud):     {n_fraud:,}")
print(f"  Imbalance ratio:     {n_nonfraud / n_fraud:.2f}:1")

# ── Guard 1: k_neighbors feasibility ────────────────────────────────────────
K_DEFAULT = 5
k_neighbors = min(K_DEFAULT, n_fraud - 1)
if k_neighbors < K_DEFAULT:
    print(f"\n⚠  Only {n_fraud} fraud cases — clipping k_neighbors {K_DEFAULT} → {k_neighbors}")
else:
    print(f"\n✓  k_neighbors={k_neighbors} feasible ({n_fraud} real fraud cases available)")

# ── Guard 2: already-balanced check ─────────────────────────────────────────
N_MAJORITY_TARGET = 30_000
SAMPLING_STRATEGY = 0.1          # minority/majority ratio target for SMOTE
current_ratio = n_fraud / N_MAJORITY_TARGET
if current_ratio >= SAMPLING_STRATEGY:
    print(f"⚠  Minority already at {current_ratio:.3f} ≥ target {SAMPLING_STRATEGY} "
          f"after Step 1 — SMOTE will be skipped; ENN still runs.")
else:
    target_minority = int(N_MAJORITY_TARGET * SAMPLING_STRATEGY)
    synthetic_needed = target_minority - n_fraud
    print(f"✓  SMOTE will generate ~{synthetic_needed:,} synthetic fraud samples "
          f"({n_fraud} real → ~{target_minority:,} target, ratio ~{SAMPLING_STRATEGY:.0%})")

# ── Step 1: Undersample majority to 30,000 ──────────────────────────────────
rus = RandomUnderSampler(
    sampling_strategy={0: N_MAJORITY_TARGET, 1: n_fraud},
    random_state=RANDOM_STATE
)
X_rus, y_rus = rus.fit_resample(X_train, y_train)
print(f"\nAfter RandomUnderSampler (Step 1):")
print(f"  Class 0: {(y_rus == 0).sum():,}  |  Class 1: {(y_rus == 1).sum():,}")
print(f"  Ratio:   {(y_rus == 0).sum() / (y_rus == 1).sum():.1f}:1")

# ── Step 2: Apply SMOTE-ENN, targeting ~10:1 ratio ──────────────────────────
smote_enn = SMOTEENN(
    random_state=RANDOM_STATE,
    sampling_strategy=SAMPLING_STRATEGY,
    smote=_SMOTE(
        random_state=RANDOM_STATE,
        k_neighbors=k_neighbors,
        sampling_strategy=SAMPLING_STRATEGY   # must match outer strategy
    ),
    enn=_ENN(n_neighbors=3)
)
X_train_balanced, y_train_balanced = smote_enn.fit_resample(X_rus, y_rus)

n_bal_0 = int((y_train_balanced == 0).sum())
n_bal_1 = int((y_train_balanced == 1).sum())

# ── Guard 3: Post-resample minority floor ───────────────────────────────────
MIN_FRAUD_FLOOR = 50
if n_bal_1 < MIN_FRAUD_FLOOR:
    raise RuntimeError(
        f"ENN removed too many fraud samples — only {n_bal_1} remain "
        f"(floor = {MIN_FRAUD_FLOOR}). Try a larger N_MAJORITY_TARGET or "
        f"smaller ENN n_neighbors to retain more synthetic fraud samples."
    )

balanced_ratio = n_bal_0 / max(n_bal_1, 1)
print(f"\nAfter SMOTE-ENN (Step 2, target ~10:1):")
print(f"  Class 0 (Non-Fraud): {n_bal_0:,}")
print(f"  Class 1 (Fraud):     {n_bal_1:,}  ({n_bal_1 - n_fraud:,} synthetic + {n_fraud} real)")
print(f"  Imbalance ratio:     {balanced_ratio:.2f}:1")
print(f"\nNew training set shape: {X_train_balanced.shape}")
print(f"\n✓ All guards passed. Test set is untouched.")
print(f"  → Report PR-AUC, F1, and Recall/Precision — not Accuracy.")

In [ ]:
# ── Section 9.2 Supplement: Seed-Stability Probe ────────────────────────────

PROBE_SEEDS = [RANDOM_STATE, RANDOM_STATE + 1, RANDOM_STATE + 2]
probe_results = []

print("Seed-stability probe for RUS + SMOTE-ENN resampling")
print("=" * 56)
print(f"{'Seed':>6}  {'Class 0 (Non-Fraud)':>20}  {'Class 1 (Fraud)':>16}  {'Ratio':>8}")
print("-" * 56)

for seed in PROBE_SEEDS:
    # Step 1: RUS with this seed
    _rus = RandomUnderSampler(
        sampling_strategy={0: N_MAJORITY_TARGET, 1: n_fraud},
        random_state=seed
    )
    _X_rus, _y_rus = _rus.fit_resample(X_train, y_train)

    # Step 2: SMOTE-ENN with the same seed
    _smote_enn = SMOTEENN(
        random_state=seed,
        sampling_strategy=SAMPLING_STRATEGY,
        smote=_SMOTE(
            random_state=seed,
            k_neighbors=k_neighbors,
            sampling_strategy=SAMPLING_STRATEGY
        ),
        enn=_ENN(n_neighbors=3)
    )
    _X_bal, _y_bal = _smote_enn.fit_resample(_X_rus, _y_rus)

    c0 = int((_y_bal == 0).sum())
    c1 = int((_y_bal == 1).sum())
    ratio = c0 / max(c1, 1)
    probe_results.append({'seed': seed, 'class_0': c0, 'class_1': c1, 'ratio': ratio})
    print(f"{seed:>6}  {c0:>20,}  {c1:>16,}  {ratio:>7.2f}:1")

print("-" * 56)

# Summary statistics across seeds
c0_vals  = [r['class_0'] for r in probe_results]
c1_vals  = [r['class_1'] for r in probe_results]
rat_vals = [r['ratio']   for r in probe_results]

print(f"\nMean ± Std across {len(PROBE_SEEDS)} seeds:")
print(f"  Class 0 (Non-Fraud): {np.mean(c0_vals):,.0f} ± {np.std(c0_vals):.1f}  "
      f"(CV = {np.std(c0_vals)/np.mean(c0_vals)*100:.2f}%)")
print(f"  Class 1 (Fraud):     {np.mean(c1_vals):,.0f} ± {np.std(c1_vals):.1f}  "
      f"(CV = {np.std(c1_vals)/np.mean(c1_vals)*100:.2f}%)")
print(f"  Ratio:               {np.mean(rat_vals):.2f} ± {np.std(rat_vals):.3f}")

CV_THRESHOLD = 2.0  # % — below this is considered stable
cv_c0 = np.std(c0_vals) / np.mean(c0_vals) * 100
cv_c1 = np.std(c1_vals) / np.mean(c1_vals) * 100

print()
if max(cv_c0, cv_c1) < CV_THRESHOLD:
    print(f"✓ Stable: CV < {CV_THRESHOLD}% for both classes across all seeds.")
    print(f"  The {N_MAJORITY_TARGET:,} majority subsample is representative;")
    print(f"  seed choice has negligible impact on dataset composition.")
else:
    print(f"⚠ CV ≥ {CV_THRESHOLD}% detected — consider increasing N_MAJORITY_TARGET")
    print(f"  or reporting mean ± std in thesis results tables.")

### 9.3 Feature Scaling

In [ ]:
# Feature Scaling
print("Scaling features using RobustScaler...")

# Use RobustScaler (better for outliers)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)

print(f"✓ Training data scaled: {X_train_scaled.shape}")
print(f"✓ Test data scaled: {X_test_scaled.shape}")

# Convert back to DataFrame for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_names)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_names)

print("\nData is ready for model training!")

### 9.4 Model Evaluation Function

Define the evaluation function before training models.

In [ ]:

from sklearn.metrics import average_precision_score

def evaluate_model(y_true, y_pred, y_pred_proba, model_name):
    """
    Comprehensive model evaluation — reports AUPRC in addition to standard metrics.
    """
    # Calculate metrics
    accuracy      = accuracy_score(y_true, y_pred)
    precision     = precision_score(y_true, y_pred)
    recall        = recall_score(y_true, y_pred)
    f1            = f1_score(y_true, y_pred)
    roc_auc       = roc_auc_score(y_true, y_pred_proba)
    avg_precision = average_precision_score(y_true, y_pred_proba)  # Area under PR curve
    mcc           = matthews_corrcoef(y_true, y_pred)

    # Print metrics
    print(f"Accuracy:      {accuracy:.4f}")
    print(f"Precision:     {precision:.4f}")
    print(f"Recall:        {recall:.4f}")
    print(f"F1-Score:      {f1:.4f}")
    print(f"ROC-AUC:       {roc_auc:.4f}")
    print(f"Avg Precision: {avg_precision:.4f}  (AUPRC — area under PR curve)")
    print(f"MCC:           {mcc:.4f}")

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    print(f"\nConfusion Matrix:")
    print(cm)

    # Classification report
    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred))

    return {
        'model':         model_name,
        'accuracy':      accuracy,
        'precision':     precision,
        'recall':        recall,
        'f1':            f1,
        'roc_auc':       roc_auc,
        'avg_precision': avg_precision,
        'mcc':           mcc,
    }

print("✓ Model evaluation function defined")


## 10. Baseline Models

In [ ]:

results_list = []

print("="*80)
print("MODEL TRAINING AND EVALUATION")
print("="*80)
print("\nInitialized results tracking...")
print(f"Test set size: {len(y_test):,} samples")
print(f"Test set fraud cases: {y_test.sum():,} ({y_test.sum()/len(y_test)*100:.2f}%)")

y_test.value_counts()

In [ ]:
# Logistic Regression Baseline Model
print("Training Logistic Regression Model...")

lr_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train_balanced)

y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

lr_results = evaluate_model(y_test, y_pred_lr, y_pred_proba_lr, "Logistic Regression")
results_list.append(lr_results)

print("\n✓ Logistic Regression training complete!")


## 11. Optimized Models

All models are tuned with **Hyperopt (TPE)** on the **SMOTE-ENN balanced training set**.  
No model-level class imbalance parameters (e.g. `scale_pos_weight`, `class_weight`) are used — class balance is handled exclusively by the SMOTE-ENN resampling step in Section 9.

Each model follows the same pipeline:
1. 80/20 stratified split of the SMOTE-ENN set for CV search + calibration
2. Hyperopt TPE search (40 evaluations, 3-fold CV scoring = ROC-AUC)
3. Isotonic / Platt calibration on the held-out val slice
4. Youden's J threshold selection on the calibrated val probabilities
5. Refit on full SMOTE-ENN set + final test-set evaluation

In [ ]:

# ═══════════════════════════════════════════════════════════════════════
# Section 11.1  Random Forest — Hyperopt Tuning  (Run 4)
# Recall-focused redesign:
#   • CV scoring → average_precision (PR-AUC): threshold-independent and
#     directly penalises missed positives; better signal than ROC-AUC on
#     heavily imbalanced data.
#   • More trees (300–700) → stronger ensemble averaging, lower variance.
#   • Shallower depth cap (≤7) → less memorisation of SMOTE synthetics.
#   • Higher min_samples_leaf/split → smoother decision boundaries.
#   • Slightly lower max_features → more diversity between trees.
# ═══════════════════════════════════════════════════════════════════════

from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
from hyperopt.pyll import scope
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve
from sklearn.model_selection import cross_val_score
import warnings

print("=" * 60)
print("Section 11.1  Hyperparameter Tuning: Random Forest")
print("=" * 60)

# ── 1. SMOTE-ENN balanced training data ──────────────────────────────────────
print(f"\nSMOTE-ENN training samples : {len(y_train_balanced):,}")
print(f"Fraud cases                : {int(y_train_balanced.sum()):,}  ({y_train_balanced.mean():.4%})")

# ── 2. Hold-out 20 % for threshold selection ─────────────────────────────────
X_tr_rf, X_val_rf, y_tr_rf, y_val_rf = train_test_split(
    X_train_scaled, y_train_balanced,
    test_size=0.20, random_state=RANDOM_STATE + 1, stratify=y_train_balanced
)
print(f"Train subset : {len(y_tr_rf):,} | Val subset : {len(y_val_rf):,}")
print(f"Val fraud    : {int(y_val_rf.sum()):,}")

# ── 3. Hyperopt search space & objective ─────────────────────────────────────
# average_precision (PR-AUC) focuses the search on recall: unlike ROC-AUC it
# is not inflated by the massive number of true negatives in the test set.
cv_rf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

space_rf = {
    # More trees → better averaging → lower prediction variance
    'n_estimators'     : scope.int(hp.quniform('n_estimators',       300, 700, 50)),
    # Cap depth at 7: deep trees memorise SMOTE synthetic patterns
    'max_depth'        : hp.choice('max_depth',         [3, 4, 5, 6, 7]),
    # Higher leaf floor smooths decision boundaries, aids generalisation
    'min_samples_leaf' : scope.int(hp.quniform('min_samples_leaf',    5,  30,  5)),
    'min_samples_split': scope.int(hp.quniform('min_samples_split',  10,  30,  5)),
    # Lower max_features ceiling → more diversity between trees
    'max_features'     : hp.choice('max_features',      ['sqrt', 0.2, 0.3, 0.4]),
    # Continuous subsample range: better coverage for Hyperopt TPE
    'max_samples'      : hp.uniform('max_samples',       0.50, 0.85),
}

def _objective_rf(params):
    m = RandomForestClassifier(**params, random_state=RANDOM_STATE, n_jobs=-1)
    scores = cross_val_score(m, X_tr_rf, y_tr_rf, cv=cv_rf,
                             scoring='average_precision', n_jobs=1)
    return {'loss': -scores.mean(), 'status': STATUS_OK}

trials_rf = Trials()
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _best_rf = fmin(
        fn=_objective_rf, space=space_rf, algo=tpe.suggest,
        max_evals=60, trials=trials_rf,
        rstate=np.random.default_rng(RANDOM_STATE), verbose=False
    )

best_rf_params = space_eval(space_rf, _best_rf)
print(f"\nBest RF params        : {best_rf_params}")
print(f"Best CV Avg-Precision : {-trials_rf.best_trial['result']['loss']:.4f}")

# ── 4. Probability calibration (isotonic, 3-fold on train subset) ────────────
calibrated_rf = CalibratedClassifierCV(
    RandomForestClassifier(**best_rf_params, random_state=RANDOM_STATE, n_jobs=-1),
    cv=3, method='isotonic')
calibrated_rf.fit(X_tr_rf, y_tr_rf)

val_proba_rf    = calibrated_rf.predict_proba(X_val_rf)[:, 1]
rf_fraud_probas = val_proba_rf[y_val_rf == 1]
print(f"\nCalibrated fraud probabilities — min: {rf_fraud_probas.min():.4f}  "
      f"max: {rf_fraud_probas.max():.4f}  mean: {rf_fraud_probas.mean():.4f}")

# ── 5. Youden's J threshold ────────────────────────────────────────────────────
fprs_rf, tprs_rf, roc_thr_rf = roc_curve(y_val_rf, val_proba_rf)
youden_rf   = tprs_rf - fprs_rf
best_idx_rf = int(np.argmax(youden_rf))
best_thr_rf = float(roc_thr_rf[best_idx_rf])
print(f"\nYouden's J threshold : {best_thr_rf:.4f}")
print(f"  Val TPR (recall)   : {tprs_rf[best_idx_rf]:.4f}")
print(f"  Val FPR            : {fprs_rf[best_idx_rf]:.4f}")

# ── 6. Refit on full SMOTE-ENN training set ───────────────────────────────────
print(f"\nRefitting best params on full SMOTE-ENN set ({len(y_train_balanced):,} rows)…")
best_rf_model = CalibratedClassifierCV(
    RandomForestClassifier(**best_rf_params, random_state=RANDOM_STATE, n_jobs=-1),
    cv=3, method='isotonic')
best_rf_model.fit(X_train_scaled, y_train_balanced)

# ── 7. Evaluate on test set ──────────────────────────────────────────────────
rf_tuned_proba   = best_rf_model.predict_proba(X_test_scaled)[:, 1]
rf_tuned_pred    = (rf_tuned_proba >= best_thr_rf).astype(int)
rf_tuned_results = evaluate_model(y_test, rf_tuned_pred, rf_tuned_proba, "Random Forest (Tuned)")
results_list = [r for r in results_list if r.get('model') != 'Random Forest (Tuned)']
results_list.append(rf_tuned_results)
print("\n✓ Random Forest tuning complete!")


In [ ]:

# ═══════════════════════════════════════════════════════════════════════
# Section 11.2  XGBoost — Hyperopt Tuning  (Run 4)
# Recall-focused redesign:
#   Run 3 problem: CV AUC 0.99 → Test Recall 16.7% — extreme SMOTE overfit.
#   Diagnosis: depth 3-5 is still too deep when combined with 500 trees and
#   weak regularization; the model memorises synthetic minority samples.
#   Fixes applied:
#   • CV scoring → average_precision (PR-AUC): focuses search on recall.
#   • Hard-cap max_depth at 2–4 (shallower = less synthetic memorisation).
#   • Reduce n_estimators ceiling to 400 (fewer trees needed with strong reg).
#   • Slow learning_rate (0.005–0.05): forces gradual, generalisable learning.
#   • min_child_weight 20–80: very high floor prevents splits on rare
#     synthetic-only patterns.
#   • gamma 3–40, reg_alpha 2–60, reg_lambda 3–60: aggressive pruning and
#     L1/L2 penalties collapse memorised leaf predictions toward the base rate.
# ═══════════════════════════════════════════════════════════════════════

from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
from hyperopt.pyll import scope
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve
from sklearn.model_selection import cross_val_score
import warnings

print("=" * 60)
print("Section 11.2  Hyperparameter Tuning: XGBoost")
print("=" * 60)

try:
    import xgboost as xgb

    # ── 1. SMOTE-ENN balanced training data ──────────────────────────────────
    print(f"\nSMOTE-ENN training samples : {len(y_train_balanced):,}")
    print(f"Fraud cases                : {int(y_train_balanced.sum()):,}  ({y_train_balanced.mean():.4%})")

    # ── 2. Hold-out 20 % for threshold selection ─────────────────────────────
    X_tr_xgb, X_val_xgb, y_tr_xgb, y_val_xgb = train_test_split(
        X_train_scaled, y_train_balanced,
        test_size=0.20, random_state=RANDOM_STATE + 1, stratify=y_train_balanced
    )
    print(f"Train subset : {len(y_tr_xgb):,} | Val subset : {len(y_val_xgb):,}")
    print(f"Val fraud    : {int(y_val_xgb.sum()):,}")

    # ── 3. Hyperopt search space & objective ─────────────────────────────────
    cv_xgb = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    space_xgb = {
        # Fewer trees — strong regularisation makes each tree count
        'n_estimators'    : scope.int(hp.quniform('n_estimators',      150, 400,  50)),
        # Hard cap at depth 4: prevents leaf-level SMOTE memorisation
        'max_depth'       : scope.int(hp.quniform('max_depth',           2,   4,   1)),
        # Slow learning → each round learns a smoother correction
        'learning_rate'   : hp.loguniform('learning_rate',  np.log(0.005), np.log(0.05)),
        'subsample'       : hp.uniform('subsample',          0.50, 0.80),
        'colsample_bytree': hp.uniform('colsample_bytree',   0.50, 0.80),
        # Very high min_child_weight — a split must be supported by many samples
        'min_child_weight': scope.int(hp.quniform('min_child_weight',   20,  80,   5)),
        # Strong gamma: aggressively prune splits that don't add much
        'gamma'           : hp.loguniform('gamma',      np.log(3.0),  np.log(40.0)),
        # Heavy L1 + L2: drive leaf weights toward zero, reduce overfit
        'reg_alpha'       : hp.loguniform('reg_alpha',  np.log(2.0),  np.log(60.0)),
        'reg_lambda'      : hp.loguniform('reg_lambda', np.log(3.0),  np.log(60.0)),
    }

    def _objective_xgb(params):
        m = xgb.XGBClassifier(
            **params,
            random_state=RANDOM_STATE, n_jobs=-1,
            tree_method='hist', eval_metric='logloss', verbosity=0
        )
        scores = cross_val_score(m, X_tr_xgb, y_tr_xgb, cv=cv_xgb,
                                 scoring='average_precision', n_jobs=1)
        return {'loss': -scores.mean(), 'status': STATUS_OK}

    trials_xgb = Trials()
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        _best_xgb = fmin(
            fn=_objective_xgb, space=space_xgb, algo=tpe.suggest,
            max_evals=60, trials=trials_xgb,
            rstate=np.random.default_rng(RANDOM_STATE), verbose=False
        )

    best_xgb_params = space_eval(space_xgb, _best_xgb)
    print(f"\nBest XGB params       : {best_xgb_params}")
    print(f"Best CV Avg-Precision : {-trials_xgb.best_trial['result']['loss']:.4f}")

    # ── 4. Probability calibration (isotonic, 3-fold on train subset) ────────
    calibrated_xgb = CalibratedClassifierCV(
        xgb.XGBClassifier(
            **best_xgb_params,
            random_state=RANDOM_STATE, n_jobs=-1,
            tree_method='hist', eval_metric='logloss', verbosity=0
        ),
        cv=3, method='isotonic')
    calibrated_xgb.fit(X_tr_xgb, y_tr_xgb)

    val_proba_xgb    = calibrated_xgb.predict_proba(X_val_xgb)[:, 1]
    xgb_fraud_probas = val_proba_xgb[y_val_xgb == 1]
    print(f"\nCalibrated fraud probabilities — min: {xgb_fraud_probas.min():.4f}  "
          f"max: {xgb_fraud_probas.max():.4f}  mean: {xgb_fraud_probas.mean():.4f}")

    # ── 5. Youden's J threshold ────────────────────────────────────────────────
    fprs_xgb, tprs_xgb, roc_thr_xgb = roc_curve(y_val_xgb, val_proba_xgb)
    youden_xgb   = tprs_xgb - fprs_xgb
    best_idx_xgb = int(np.argmax(youden_xgb))
    best_thr_xgb = float(roc_thr_xgb[best_idx_xgb])
    print(f"\nYouden's J threshold : {best_thr_xgb:.4f}")
    print(f"  Val TPR (recall)   : {tprs_xgb[best_idx_xgb]:.4f}")
    print(f"  Val FPR            : {fprs_xgb[best_idx_xgb]:.4f}")

    # ── 6. Refit on full SMOTE-ENN training set ───────────────────────────────
    print(f"\nRefitting best params on full SMOTE-ENN set ({len(y_train_balanced):,} rows)…")
    best_xgb_model = CalibratedClassifierCV(
        xgb.XGBClassifier(
            **best_xgb_params,
            random_state=RANDOM_STATE, n_jobs=-1,
            tree_method='hist', eval_metric='logloss', verbosity=0
        ),
        cv=3, method='isotonic')
    best_xgb_model.fit(X_train_scaled, y_train_balanced)

    # ── 7. Evaluate on test set ──────────────────────────────────────────────
    xgb_tuned_proba   = best_xgb_model.predict_proba(X_test_scaled)[:, 1]
    xgb_tuned_pred    = (xgb_tuned_proba >= best_thr_xgb).astype(int)
    xgb_tuned_results = evaluate_model(y_test, xgb_tuned_pred, xgb_tuned_proba, "XGBoost (Tuned)")
    results_list = [r for r in results_list if r.get('model') != 'XGBoost (Tuned)']
    results_list.append(xgb_tuned_results)
    print("\n✓ XGBoost tuning complete!")

except ImportError:
    print("⚠  XGBoost not available — skipping Section 11.2.")


In [ ]:

# ═══════════════════════════════════════════════════════════════════════
# Section 11.3  LightGBM — Hyperopt Tuning  (Run 4)
# Recall-focused redesign:
#   Run 3 problem: CV AUC 0.996 → Test Recall 11.1% — worst SMOTE overfit.
#   Diagnosis: num_leaves=44 with max_depth=-1 allows unlimited tree depth;
#   min_child_samples=130 is still too low for 3,000 synthetic fraud samples.
#   Fixes applied:
#   • CV scoring → average_precision (PR-AUC).
#   • Hard-cap num_leaves to 8–31 (31 = 2^5, equivalent to depth-5 tree).
#   • Remove max_depth=-1; force depth [3–6] explicitly.
#   • min_child_samples 200–600: a leaf must be supported by at least
#     200 training samples — eliminates synthetic-only leaves.
#   • lambda_l2 5–100: very strong L2 collapses leaf weights toward zero.
#   • lambda_l1 0.1–20: L1 sparsifies the tree structure.
#   • Reduce n_estimators ceiling to 350.
# ═══════════════════════════════════════════════════════════════════════

from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
from hyperopt.pyll import scope
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve
from sklearn.model_selection import cross_val_score
import warnings

try:
    import lightgbm as lgb

    print("=" * 60)
    print("Section 11.3  Hyperparameter Tuning: LightGBM")
    print("=" * 60)

    # ── 1. SMOTE-ENN balanced training data ──────────────────────────────────
    print(f"\nSMOTE-ENN training samples : {len(y_train_balanced):,}")
    print(f"Fraud cases                : {int(y_train_balanced.sum()):,}  ({y_train_balanced.mean():.4%})")

    # ── 2. Hold-out 20 % for threshold selection ─────────────────────────────
    X_tr_lgb, X_val_lgb, y_tr_lgb, y_val_lgb = train_test_split(
        X_train_scaled, y_train_balanced,
        test_size=0.20, random_state=RANDOM_STATE + 1, stratify=y_train_balanced
    )
    print(f"Train subset : {len(y_tr_lgb):,} | Val subset : {len(y_val_lgb):,}")
    print(f"Val fraud    : {int(y_val_lgb.sum()):,}")

    # ── 3. Hyperopt search space & objective ─────────────────────────────────
    cv_lgb = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    space_lgb = {
        # Fewer trees with strong regularisation
        'n_estimators'     : scope.int(hp.quniform('n_estimators',        100, 350, 50)),
        # Hard depth cap − no unlimited trees (max_depth=-1 was the root cause)
        'max_depth'        : hp.choice('max_depth',         [3, 4, 5, 6]),
        # Slow learning to avoid rapid SMOTE memorisation
        'learning_rate'    : hp.loguniform('learning_rate',  np.log(0.01), np.log(0.10)),
        # Hard cap on leaves: 31 = depth-5 equivalent; prevents wide flat trees
        'num_leaves'       : scope.int(hp.quniform('num_leaves',            8,  31,  4)),
        'subsample'        : hp.uniform('subsample',          0.60, 1.0),
        'colsample_bytree' : hp.uniform('colsample_bytree',   0.60, 1.0),
        # High min_child_samples: each leaf needs real support, not just SMOTE
        'min_child_samples': scope.int(hp.quniform('min_child_samples',   200, 600, 50)),
        # Aggressive L1/L2 regularisation
        'lambda_l1'        : hp.loguniform('lambda_l1',   np.log(0.1),   np.log(20.0)),
        'lambda_l2'        : hp.loguniform('lambda_l2',   np.log(5.0),  np.log(100.0)),
    }

    def _objective_lgb(params):
        m = lgb.LGBMClassifier(
            **params, random_state=RANDOM_STATE, n_jobs=-1,
            bagging_freq=1, verbosity=-1
        )
        scores = cross_val_score(m, X_tr_lgb, y_tr_lgb, cv=cv_lgb,
                                 scoring='average_precision', n_jobs=1)
        return {'loss': -scores.mean(), 'status': STATUS_OK}

    trials_lgb = Trials()
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        _best_lgb = fmin(
            fn=_objective_lgb, space=space_lgb, algo=tpe.suggest,
            max_evals=60, trials=trials_lgb,
            rstate=np.random.default_rng(RANDOM_STATE), verbose=False
        )

    best_lgb_params = space_eval(space_lgb, _best_lgb)
    print(f"\nBest LGB params       : {best_lgb_params}")
    print(f"Best CV Avg-Precision : {-trials_lgb.best_trial['result']['loss']:.4f}")

    # ── 4. Isotonic calibration (3-fold on train subset) ─────────────────────
    calibrated_lgb = CalibratedClassifierCV(
        lgb.LGBMClassifier(
            **best_lgb_params, random_state=RANDOM_STATE, n_jobs=-1,
            bagging_freq=1, verbosity=-1
        ),
        cv=3, method='isotonic')
    calibrated_lgb.fit(X_tr_lgb, y_tr_lgb)

    val_proba_lgb    = calibrated_lgb.predict_proba(X_val_lgb)[:, 1]
    lgb_fraud_probas = val_proba_lgb[y_val_lgb == 1]
    print(f"\nCalibrated fraud probabilities — min: {lgb_fraud_probas.min():.4f}  "
          f"max: {lgb_fraud_probas.max():.4f}  mean: {lgb_fraud_probas.mean():.4f}")

    # ── 5. Youden's J threshold ────────────────────────────────────────────────
    fprs_lgb, tprs_lgb, roc_thr_lgb = roc_curve(y_val_lgb, val_proba_lgb)
    youden_lgb   = tprs_lgb - fprs_lgb
    best_idx_lgb = int(np.argmax(youden_lgb))
    best_thr_lgb = float(roc_thr_lgb[best_idx_lgb])
    print(f"\nYouden's J threshold : {best_thr_lgb:.4f}")
    print(f"  Val TPR (recall)   : {tprs_lgb[best_idx_lgb]:.4f}")
    print(f"  Val FPR            : {fprs_lgb[best_idx_lgb]:.4f}")

    # ── 6. Refit on full SMOTE-ENN training set ───────────────────────────────
    print(f"\nRefitting best params on full SMOTE-ENN set ({len(y_train_balanced):,} rows)…")
    best_lgb_model = CalibratedClassifierCV(
        lgb.LGBMClassifier(
            **best_lgb_params, random_state=RANDOM_STATE, n_jobs=-1,
            bagging_freq=1, verbosity=-1
        ),
        cv=3, method='isotonic')
    best_lgb_model.fit(X_train_scaled, y_train_balanced)

    # ── 7. Evaluate on test set ──────────────────────────────────────────────
    lgb_tuned_proba   = best_lgb_model.predict_proba(X_test_scaled)[:, 1]
    lgb_tuned_pred    = (lgb_tuned_proba >= best_thr_lgb).astype(int)
    lgb_tuned_results = evaluate_model(y_test, lgb_tuned_pred, lgb_tuned_proba, "LightGBM (Tuned)")
    results_list = [r for r in results_list if r.get('model') != 'LightGBM (Tuned)']
    results_list.append(lgb_tuned_results)
    print("\n✓ LightGBM tuning complete!")

except ImportError:
    print("⚠  LightGBM not available — skipping Section 11.3.")


In [ ]:

# ═══════════════════════════════════════════════════════════════════════
# Section 11.4  HistGradientBoostingClassifier — Hyperopt Tuning  (Run 4)
# Recall-focused redesign:
#   Run 3 problem: CV AUC 0.996 → Test Recall 7.4% — catastrophic overfit.
#   Diagnosis: max_leaf_nodes=60 + max_depth=15 + l2≈2 → very expressive
#   trees that produce near-zero probabilities for all test fraud cases.
#   Fixes applied:
#   • CV scoring → average_precision (PR-AUC).
#   • Hard-cap max_leaf_nodes to 8–24 (equivalent to depth ≤ 4–5).
#   • Hard-cap max_depth to [3, 4, 5] — remove None and deep options.
#   • Increase min_samples_leaf to 150–500: leaves with few samples
#     represent SMOTE artefacts, not real fraud patterns.
#   • l2_regularization 5–100: collapse shallow tree leaf weights.
#   • Slow learning_rate 0.005–0.04: prevents aggressive SMOTE fitting.
#   • max_bins 32–64: coarser splits → less ability to overfit.
#   • Reduce max_iter ceiling to 400.
# ═══════════════════════════════════════════════════════════════════════

from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
from hyperopt.pyll import scope
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve
from sklearn.model_selection import cross_val_score
import warnings

print("=" * 60)
print("Section 11.4  Hyperparameter Tuning: HistGradientBoosting")
print("=" * 60)

# ── 1. SMOTE-ENN balanced training data ──────────────────────────────────────
print(f"\nSMOTE-ENN training samples : {len(y_train_balanced):,}")
print(f"Fraud cases                : {int(y_train_balanced.sum()):,}  ({y_train_balanced.mean():.4%})")

# ── 2. Hold-out 20 % for threshold selection ─────────────────────────────────
X_tr_hgb, X_val_hgb, y_tr_hgb, y_val_hgb = train_test_split(
    X_train_scaled, y_train_balanced,
    test_size=0.20, random_state=RANDOM_STATE + 1, stratify=y_train_balanced
)
print(f"Train subset : {len(y_tr_hgb):,} | Val subset : {len(y_val_hgb):,}")
print(f"Val fraud    : {int(y_val_hgb.sum()):,}")

# ── 3. Hyperopt search space & objective ─────────────────────────────────────
cv_hgb = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

space_hgb = {
    # Fewer iterations — with strong reg they converge faster
    'max_iter'         : scope.int(hp.quniform('max_iter',           150, 400,  50)),
    # Hard depth cap: no deep trees, no None
    'max_depth'        : hp.choice('max_depth',          [3, 4, 5]),
    # Very slow learning: each stage makes a small, safe correction
    'learning_rate'    : hp.loguniform('learning_rate',  np.log(0.005), np.log(0.04)),
    # Hard cap on leaves: 24 ~ depth-4 tree
    'max_leaf_nodes'   : scope.int(hp.quniform('max_leaf_nodes',       8,  24,   4)),
    # High min_samples_leaf: prevents SMOTE-artefact leaves
    'min_samples_leaf' : scope.int(hp.quniform('min_samples_leaf',   150, 500,  50)),
    # Very strong L2: collapses leaf predictions toward the base rate
    'l2_regularization': hp.loguniform('l2_regularization', np.log(5.0), np.log(100.0)),
    # Coarser bins → fewer possible split points → less overfit
    'max_bins'         : hp.choice('max_bins', [32, 64]),
}

def _objective_hgb(params):
    m = HistGradientBoostingClassifier(
        **params, random_state=RANDOM_STATE,
        early_stopping=False, verbose=0
    )
    scores = cross_val_score(m, X_tr_hgb, y_tr_hgb, cv=cv_hgb,
                             scoring='average_precision', n_jobs=-1)
    return {'loss': -scores.mean(), 'status': STATUS_OK}

trials_hgb = Trials()
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _best_hgb = fmin(
        fn=_objective_hgb, space=space_hgb, algo=tpe.suggest,
        max_evals=60, trials=trials_hgb,
        rstate=np.random.default_rng(RANDOM_STATE), verbose=False
    )

best_hgb_params = space_eval(space_hgb, _best_hgb)
print(f"\nBest HGB params       : {best_hgb_params}")
print(f"Best CV Avg-Precision : {-trials_hgb.best_trial['result']['loss']:.4f}")

# ── 4. Isotonic calibration (3-fold on train subset) ─────────────────────────
calibrated_hgb = CalibratedClassifierCV(
    HistGradientBoostingClassifier(
        **best_hgb_params, random_state=RANDOM_STATE,
        early_stopping=False, verbose=0
    ),
    cv=3, method='isotonic')
calibrated_hgb.fit(X_tr_hgb, y_tr_hgb)

val_proba_hgb    = calibrated_hgb.predict_proba(X_val_hgb)[:, 1]
hgb_fraud_probas = val_proba_hgb[y_val_hgb == 1]
print(f"\nCalibrated fraud probabilities — min: {hgb_fraud_probas.min():.4f}  "
      f"max: {hgb_fraud_probas.max():.4f}  mean: {hgb_fraud_probas.mean():.4f}")

# ── 5. Youden's J threshold ────────────────────────────────────────────────────
fprs_hgb, tprs_hgb, roc_thr_hgb = roc_curve(y_val_hgb, val_proba_hgb)
youden_hgb   = tprs_hgb - fprs_hgb
best_idx_hgb = int(np.argmax(youden_hgb))
best_thr_hgb = float(roc_thr_hgb[best_idx_hgb])
print(f"\nYouden's J threshold : {best_thr_hgb:.4f}")
print(f"  Val TPR (recall)   : {tprs_hgb[best_idx_hgb]:.4f}")
print(f"  Val FPR            : {fprs_hgb[best_idx_hgb]:.4f}")

# ── 6. Refit on full SMOTE-ENN training set ───────────────────────────────────
print(f"\nRefitting best params on full SMOTE-ENN set ({len(y_train_balanced):,} rows)…")
best_hgb_model = CalibratedClassifierCV(
    HistGradientBoostingClassifier(
        **best_hgb_params, random_state=RANDOM_STATE,
        early_stopping=False, verbose=0
    ),
    cv=3, method='isotonic')
best_hgb_model.fit(X_train_scaled, y_train_balanced)

# ── 7. Evaluate on test set ──────────────────────────────────────────────────
hgb_tuned_proba   = best_hgb_model.predict_proba(X_test_scaled)[:, 1]
hgb_tuned_pred    = (hgb_tuned_proba >= best_thr_hgb).astype(int)
hgb_tuned_results = evaluate_model(
    y_test, hgb_tuned_pred, hgb_tuned_proba, "HistGradBoost (Tuned)"
)
results_list = [r for r in results_list if r.get('model') != 'HistGradBoost (Tuned)']
results_list.append(hgb_tuned_results)
print("\n✓ HistGradientBoosting tuning complete!")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Section 11.5  SVM (LinearSVC) — Hyperopt Tuning  (Run 6)
# Recall-focused redesign:
#   Run 5 problem: Shifting C space to [1, 100] had no effect — under
#   roc_auc scoring C=10.94 is the stable optimum regardless of range;
#   objective function is decoupled from recall → 39/54 fraud again.
#   Root cause: roc_auc on SMOTE-balanced CV data peaks at C≈10.94 and
#   is insensitive to recall at that imbalance ratio.
#   Fixes applied:
#   • CV scoring changed from roc_auc → F2 (fbeta, β=2): weighs recall
#     4× over precision; steering C toward more aggressive classifiers.
#   • C search space: loguniform [0.001, 1000] — very wide; F2 objective
#     should pull the optimum to a different (likely higher) C.
#   • Threshold selection: precision-recall curve targeting ≥85% recall
#     on the SMOTE-balanced val set (replaces Youden's J which was
#     calibrated for AUC balance, not recall maximisation).
#   • max_evals increased to 80 for better TPE exploration.
# ═══════════════════════════════════════════════════════════════════════

import joblib
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import fbeta_score, make_scorer, precision_recall_curve
from sklearn.model_selection import cross_val_score, train_test_split
import warnings

print("=" * 60)
print("Section 11.5  Hyperparameter Tuning: SVM (LinearSVC)")
print("=" * 60)

# ── 1. SMOTE-ENN balanced training data ──────────────────────────────────────
print(f"\nSMOTE-ENN training samples : {len(y_train_balanced):,}")
print(f"Fraud cases                : {int(y_train_balanced.sum()):,}  ({y_train_balanced.mean():.4%})")

# ── 2. Hold-out 20 % for calibration + threshold tuning ──────────────────────
X_tr_svm, X_val_svm, y_tr_svm, y_val_svm = train_test_split(
    X_train_scaled, y_train_balanced,
    test_size=0.20, random_state=RANDOM_STATE + 1, stratify=y_train_balanced
)
print(f"Train subset : {len(y_tr_svm):,} | Val subset : {len(y_val_svm):,}")
print(f"Val fraud    : {int(y_val_svm.sum()):,}")

# ── 3. Hyperopt: C search, scored by F2 (recall-weighted) ────────────────────
cv_svm   = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
f2_scorer = make_scorer(fbeta_score, beta=2, zero_division=0)

space_svm = {
    'C': hp.loguniform('C', np.log(0.001), np.log(1000.0)), # c=0.01, 20, C=[1.0, 100.0] then 39
} 

def _objective_svm(params):
    m = LinearSVC(
        C=params['C'], loss='squared_hinge',
        class_weight='balanced', max_iter=5000, random_state=RANDOM_STATE
    )
    scores = cross_val_score(m, X_tr_svm, y_tr_svm, cv=cv_svm,
                             scoring=f2_scorer, n_jobs=-1)
    return {'loss': -scores.mean(), 'status': STATUS_OK}

trials_svm = Trials()
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _best_svm = fmin(
        fn=_objective_svm, space=space_svm, algo=tpe.suggest,
        max_evals=80, trials=trials_svm,
        rstate=np.random.default_rng(RANDOM_STATE), verbose=False
    )

best_svm_params = space_eval(space_svm, _best_svm)
print(f"\nBest SVM params  : C={best_svm_params['C']:.5f}")
print(f"Best CV F2-score : {-trials_svm.best_trial['result']['loss']:.4f}")

# ── 4. Platt (sigmoid) calibration — 3-fold on train subset ──────────────────
calibrated_svm = CalibratedClassifierCV(
    LinearSVC(
        C=best_svm_params['C'], loss='squared_hinge',
        class_weight='balanced', max_iter=5000, random_state=RANDOM_STATE
    ),
    cv=3, method='sigmoid')
calibrated_svm.fit(X_tr_svm, y_tr_svm)

val_proba_svm    = calibrated_svm.predict_proba(X_val_svm)[:, 1]
svm_fraud_probas = val_proba_svm[y_val_svm == 1]
print(f"\nCalibrated fraud probabilities — min: {svm_fraud_probas.min():.4f}  "
      f"max: {svm_fraud_probas.max():.4f}  mean: {svm_fraud_probas.mean():.4f}")

# ── 5. Recall-targeted threshold (≥85% recall on val set) ────────────────────
RECALL_TARGET = 0.85
_, recall_vals, pr_thresholds = precision_recall_curve(y_val_svm, val_proba_svm)
# pr_thresholds has len n-1; recall_vals[i] corresponds to pr_thresholds[i]
eligible = pr_thresholds[recall_vals[:-1] >= RECALL_TARGET]
if len(eligible) > 0:
    best_thr_svm = float(eligible.max())   # highest threshold still giving ≥85% recall
else:
    best_thr_svm = float(pr_thresholds[np.argmax(recall_vals[:-1])])  # fallback
val_pred_at_thr = (val_proba_svm >= best_thr_svm).astype(int)
val_recall_at_thr = float((val_pred_at_thr[y_val_svm == 1]).mean())
print(f"\nRecall target         : {RECALL_TARGET:.0%}")
print(f"Selected threshold    : {best_thr_svm:.4f}")
print(f"Val recall at thresh  : {val_recall_at_thr:.4f}")

# ── 6. Refit on full SMOTE-ENN training set ───────────────────────────────────
print(f"\nRefitting on full SMOTE-ENN set ({len(y_train_balanced):,} rows)…")
best_svm_model = CalibratedClassifierCV(
    LinearSVC(
        C=best_svm_params['C'], loss='squared_hinge',
        class_weight='balanced', max_iter=5000, random_state=RANDOM_STATE
    ),
    cv=3, method='sigmoid')
best_svm_model.fit(X_train_scaled, y_train_balanced)

# ── 7. Evaluate on held-out test set ─────────────────────────────────────────
svm_tuned_proba   = best_svm_model.predict_proba(X_test_scaled)[:, 1]
svm_tuned_pred    = (svm_tuned_proba >= best_thr_svm).astype(int)
svm_tuned_results = evaluate_model(y_test, svm_tuned_pred, svm_tuned_proba, "SVM (Tuned)")
results_list = [r for r in results_list if r.get('model') != 'SVM (Tuned)']
results_list.append(svm_tuned_results)
print("\n✓ SVM tuning complete!")

# ── 8. Save model, scaler, threshold, and balanced training data to disk ──────
# The scaler was fitted on X_train_balanced (SMOTE-ENN data) — must be saved
# and reloaded in downstream notebooks so X_test is scaled identically.
# X_train_balanced / y_train_balanced are also saved so downstream notebooks
# can train the SHAP LinearSVC on the same data the production model saw.
MODELS_DIR = Path('/Users/roya/Desktop/Thesis/Thesis_Project/ML/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATASETS_DIR = Path('/Users/roya/Desktop/Thesis/Thesis_Project/ML/datasets')

joblib.dump(best_svm_model,                                          MODELS_DIR / 'svm_tuned_model.joblib')
joblib.dump(scaler,                                                  MODELS_DIR / 'svm_scaler.joblib')
joblib.dump({'threshold': best_thr_svm, 'C': best_svm_params['C']}, MODELS_DIR / 'svm_tuned_params.joblib')

X_train_balanced.to_csv(DATASETS_DIR / 'X_train_balanced.csv', index=True, index_label='balanced_index')
y_train_balanced.to_csv(DATASETS_DIR / 'y_train_balanced.csv', index=True, index_label='balanced_index')

print(f"\nModel saved           : {MODELS_DIR / 'svm_tuned_model.joblib'}")
print(f"Scaler saved          : {MODELS_DIR / 'svm_scaler.joblib'}  (fitted on SMOTE-ENN X_train_balanced)")
print(f"Params saved          : {MODELS_DIR / 'svm_tuned_params.joblib'}")
print(f"X_train_balanced saved: {DATASETS_DIR / 'X_train_balanced.csv'}  shape={X_train_balanced.shape}")
print(f"y_train_balanced saved: {DATASETS_DIR / 'y_train_balanced.csv'}  shape={y_train_balanced.shape}")


In [ ]:

# ═══════════════════════════════════════════════════════════════════════
# Section 11.6  Neural Network (MLPClassifier) — Hyperopt Tuning  (Run 4)
# Recall-focused redesign:
#   Run 3 problem: CV AUC 0.982 → Test Recall 14.8%.
#   Diagnosis: large networks (256-128-64) with low alpha overfit SMOTE;
#   high learning_rate_init can memorise minority class quickly.
#   Fixes applied:
#   • CV scoring → average_precision (PR-AUC).
#   • Add compact architectures: (64,32), (64,64), (32,32,32) — smaller
#     networks generalise better on this 39-feature dataset.
#   • Raise alpha range to [0.001, 0.2]: stronger L2 weight decay
#     prevents the network from memorising synthetic patterns.
#   • Tighten learning_rate_init to [3e-4, 3e-3]: avoids overfit from
#     aggressive early updates on SMOTE-heavy batches.
#   • Threshold selection via 3-way comparison: F1-max, F2-max, Youden's J.
# ═══════════════════════════════════════════════════════════════════════

from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
from hyperopt.pyll import scope
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve, precision_recall_curve
from sklearn.model_selection import cross_val_score
import warnings

print("=" * 60)
print("Section 11.6  Hyperparameter Tuning: Neural Network (MLP)")
print("=" * 60)

# ── 1. Data info ──────────────────────────────────────────────────────────────
print(f"\nBalanced training set (SMOTE-ENN) : {len(y_train_balanced):,} samples")
print(f"  Class 0 : {int((y_train_balanced==0).sum()):,}  |  "
      f"Class 1 : {int((y_train_balanced==1).sum()):,}")
print(f"Full training samples : {len(y_train):,}  (original, pre-SMOTE)")
print(f"Fraud cases           : {int(y_train.sum()):,}  ({y_train.mean():.4%})")

# ── 2. Hold-out 20 % of BALANCED data for threshold derivation ───────────────
# Consistent with RF/XGB/LGB/HGB/SVM: split the balanced SMOTE set (~3,000 fraud).
# Original unbalanced split gives only ~43 fraud cases in val — far too few to
# reliably optimise F1-max/F2-max/Youden's J thresholds.  Balanced split gives
# ~600 fraud cases, matching every other Section 11 model.
X_tr_mlp, X_val_mlp, y_tr_mlp_orig, y_val_mlp = train_test_split(
    X_train_scaled, y_train_balanced,
    test_size=0.20, random_state=RANDOM_STATE + 5, stratify=y_train_balanced
)
print(f"\nThreshold val subset : {len(y_tr_mlp_orig):,} | {len(y_val_mlp):,}")
print(f"Val fraud            : {int(y_val_mlp.sum()):,}")

# ── 3. CV search on train split (subsample for speed) ────────────────────────
# Use the 80 % training split as the CV population (mirrors RF/XGB pattern).
_MLP_CV_CAP = 20_000
if len(y_tr_mlp_orig) > _MLP_CV_CAP:
    _rng = np.random.RandomState(RANDOM_STATE)
    _idx = _rng.choice(len(y_tr_mlp_orig), size=_MLP_CV_CAP, replace=False)
    if hasattr(X_tr_mlp, 'iloc'):
        X_mlp_cv = X_tr_mlp.iloc[_idx]
        y_mlp_cv = y_tr_mlp_orig.iloc[_idx]
    else:
        X_mlp_cv = X_tr_mlp[_idx]
        y_mlp_cv = y_tr_mlp_orig[_idx]
else:
    X_mlp_cv = X_tr_mlp
    y_mlp_cv = y_tr_mlp_orig

print(f"\n⚡ CV search subsample : {len(y_mlp_cv):,} rows  "
      f"(fraud: {int((y_mlp_cv==1).sum()):,}, {(y_mlp_cv==1).mean():.4%})")

cv_mlp = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

space_mlp = {
    # Mix of sizes: compact networks generalise better on this 39-feature set
    'hidden_layer_sizes': hp.choice('hidden_layer_sizes', [
        (64, 32),           # compact — low overfit risk
        (64, 64),           # compact symmetric
        (32, 32, 32),       # deep but narrow
        (128, 64),          # moderate
        (128, 64, 32),      # moderate tapered
        (256, 128, 64),     # large (kept for comparison)
        (128, 128),         # moderate symmetric
    ]),
    # Higher alpha floor: stronger L2 weight decay prevents SMOTE memorisation
    'alpha'             : hp.loguniform('alpha',              np.log(1e-3), np.log(0.20)),
    # Tighter lr range: avoids aggressive early memorisation of minority class
    'learning_rate_init': hp.loguniform('learning_rate_init', np.log(3e-4), np.log(3e-3)),
    'batch_size'        : hp.choice('batch_size', [128, 256, 512]),
}

def _objective_mlp(params):
    m = MLPClassifier(
        **params,
        activation='relu', solver='adam',
        learning_rate='adaptive',
        max_iter=300, random_state=RANDOM_STATE,
        early_stopping=True, validation_fraction=0.10,
        n_iter_no_change=15, verbose=False,
    )
    scores = cross_val_score(m, X_mlp_cv, y_mlp_cv, cv=cv_mlp,
                             scoring='average_precision', n_jobs=1)
    return {'loss': -scores.mean(), 'status': STATUS_OK}

trials_mlp = Trials()
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _best_mlp = fmin(
        fn=_objective_mlp, space=space_mlp, algo=tpe.suggest,
        max_evals=40, trials=trials_mlp,
        rstate=np.random.default_rng(RANDOM_STATE), verbose=False
    )

best_mlp_params = space_eval(space_mlp, _best_mlp)
print(f"\nBest MLP params       : {best_mlp_params}")
print(f"Best CV Avg-Precision : {-trials_mlp.best_trial['result']['loss']:.4f}")

# ── 4. Sigmoid (Platt) calibration (3-fold on balanced training set) ─────────
calibrated_mlp = CalibratedClassifierCV(
    MLPClassifier(
        **best_mlp_params,
        activation='relu', solver='adam',
        learning_rate='adaptive',
        max_iter=300, random_state=RANDOM_STATE,
        early_stopping=True, validation_fraction=0.10,
        n_iter_no_change=15, verbose=False,
    ),
    cv=3, method='sigmoid')
calibrated_mlp.fit(X_train_scaled, y_train_balanced)

val_proba_mlp    = calibrated_mlp.predict_proba(X_val_mlp)[:, 1]
mlp_fraud_probas = val_proba_mlp[y_val_mlp == 1]
print(f"\nCalibrated fraud probabilities — min: {mlp_fraud_probas.min():.4f}  "
      f"max: {mlp_fraud_probas.max():.4f}  mean: {mlp_fraud_probas.mean():.4f}")

# ── 5. Three-threshold comparison on validation set ─────────────────────────
# Three candidates are evaluated; best by validation F2 score is selected.
# F2 (β=2) weights recall twice as heavily as precision — appropriate for fraud.

# F1-max and F2-max thresholds derived from the PR curve
_pr_prec, _pr_rec, _pr_thr = precision_recall_curve(y_val_mlp, val_proba_mlp)
_p, _r = _pr_prec[:-1], _pr_rec[:-1]   # last sentinel point excluded

_f1_vals   = np.where(_p + _r > 0, 2 * _p * _r / (_p + _r), 0.0)
thr_f1_mlp = float(_pr_thr[int(np.argmax(_f1_vals))])

_denom_f2  = 4 * _p + _r
_f2_vals   = np.where(_denom_f2 > 0, 5 * _p * _r / _denom_f2, 0.0)
thr_f2_mlp = float(_pr_thr[int(np.argmax(_f2_vals))])

# Youden's J threshold derived from the ROC curve
fprs_mlp, tprs_mlp, roc_thr_mlp = roc_curve(y_val_mlp, val_proba_mlp)
youden_mlp     = tprs_mlp - fprs_mlp
best_idx_mlp   = int(np.argmax(youden_mlp))
thr_youden_mlp = float(roc_thr_mlp[best_idx_mlp])

# Evaluate each candidate on the validation set; collect results
_candidates = {
    'F1-max'     : thr_f1_mlp,
    'F2-max'     : thr_f2_mlp,
    "Youden's J" : thr_youden_mlp,
}
_cmp_results = {}
best_thr_mlp = None
_best_val_f2 = -1.0
_best_label  = ''

for _label, _thr in _candidates.items():
    _pv  = (val_proba_mlp >= _thr).astype(int)
    _tp  = int(((_pv == 1) & (y_val_mlp == 1)).sum())
    _fp  = int(((_pv == 1) & (y_val_mlp == 0)).sum())
    _fn  = int(((_pv == 0) & (y_val_mlp == 1)).sum())
    _vr  = _tp / (_tp + _fn) if (_tp + _fn) > 0 else 0.0
    _vp  = _tp / (_tp + _fp) if (_tp + _fp) > 0 else 0.0
    _vf2 = 5 * _vp * _vr / (4 * _vp + _vr) if (4 * _vp + _vr) > 0 else 0.0
    _cmp_results[_label] = (_thr, _vr, _vp, _vf2)
    if _vf2 > _best_val_f2:
        _best_val_f2 = _vf2
        best_thr_mlp = _thr
        _best_label  = _label

print("\n─── Threshold Comparison (Validation Set) ──────────────────────────")
print(f"{'Criterion':<14} {'Threshold':>10}  {'Val Recall':>10}  {'Val Prec':>9}  {'Val F2':>8}")
print("─" * 60)
for _label, (_thr, _vr, _vp, _vf2) in _cmp_results.items():
    _mark = "  ★ selected" if _label == _best_label else ""
    print(f"{_label:<14} {_thr:>10.4f}  {_vr:>10.4f}  {_vp:>9.4f}  {_vf2:>8.4f}{_mark}")
print("─" * 60)
print(f"\n✓ Selected: {_best_label} = {best_thr_mlp:.4f}  (val F2 = {_best_val_f2:.4f})")

# ── 6. Refit best architecture on full balanced set with longer budget ─────────
print(f"\nRefitting best params on full balanced set ({len(y_train_balanced):,} rows)…")
best_mlp_model = CalibratedClassifierCV(
    MLPClassifier(
        **best_mlp_params,
        activation='relu', solver='adam',
        learning_rate='adaptive',
        max_iter=500, random_state=RANDOM_STATE,
        early_stopping=True, validation_fraction=0.10,
        n_iter_no_change=20, verbose=False,
    ),
    cv=3, method='sigmoid')
best_mlp_model.fit(X_train_scaled, y_train_balanced)

# ── 7. Evaluate on held-out test set ─────────────────────────────────────────
mlp_tuned_proba   = best_mlp_model.predict_proba(X_test_scaled)[:, 1]
mlp_tuned_pred    = (mlp_tuned_proba >= best_thr_mlp).astype(int)
mlp_tuned_results = evaluate_model(y_test, mlp_tuned_pred, mlp_tuned_proba, "MLP (Tuned)")
results_list = [r for r in results_list if r.get('model') != 'MLP (Tuned)']
results_list.append(mlp_tuned_results)
print("\n✓ MLP tuning complete!")


### 11.7 Graph Neural Network — GraphSAGE  

In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# Section 14.1g  Graph Neural Network — GraphSAGE 
# ═══════════════════════════════════════════════════════════════════════════════


import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
from sklearn.neighbors import kneighbors_graph, NearestNeighbors
from sklearn.linear_model import LogisticRegression as _CalLR
from sklearn.metrics import roc_curve

# ── Minimal Adam (avoids torch.optim entirely — no torch._dynamo import issues) ──
class _Adam:
    """Pure-Python Adam optimizer; sidesteps torch._dynamo import in torch.optim."""
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0):
        self.params = [p for p in params if p.requires_grad]
        self.lr = lr; self.b1, self.b2 = betas
        self.eps = eps; self.wd = weight_decay; self.t = 0
        self.m = [torch.zeros_like(p.data) for p in self.params]
        self.v = [torch.zeros_like(p.data) for p in self.params]

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.detach_()
                p.grad.zero_()

    def step(self):
        self.t += 1
        bc1, bc2 = 1 - self.b1 ** self.t, 1 - self.b2 ** self.t
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            g = p.grad.data
            if self.wd:
                g = g.add(p.data, alpha=self.wd)
            self.m[i].mul_(self.b1).add_(g, alpha=1 - self.b1)
            self.v[i].mul_(self.b2).addcmul_(g, g, value=1 - self.b2)
            m_hat = self.m[i] / bc1
            v_hat = self.v[i] / bc2
            p.data.addcdiv_(m_hat, v_hat.sqrt().add_(self.eps), value=-self.lr)

print("=" * 60)
print("Section 14.1g  Graph Neural Network (GraphSAGE")
print("=" * 60)
print(f"  PyTorch {torch.__version__}  (pure-torch, no torch_geometric needed)")

# ── 0. Device ──────────────────────────────────────────────────────────────────
_dev = (torch.device("mps")  if torch.backends.mps.is_available() else
        torch.device("cuda") if torch.cuda.is_available() else
        torch.device("cpu"))
print(f"  Device : {_dev}")

# ── 1. Data arrays ─────────────────────────────────────────────────────────────
def _np32(s): return (s.values if hasattr(s,'values') else np.array(s)).astype('float32')
def _np64(s): return (s.values if hasattr(s,'values') else np.array(s)).astype('int64')

_X_tr = _np32(X_train_scaled);   _y_tr = _np64(y_train_balanced)
_X_te = _np32(X_test_scaled);    _y_te = _np64(y_test)
if 'X_train_orig_scaled' not in dir():
    X_train_orig_scaled = pd.DataFrame(
        scaler.transform(X_train), columns=feature_names, index=X_train.index
    )
_Xo   = _np32(X_train_orig_scaled); _yo = _np64(y_train)  # original unbalanced training data
N_TR, N_FEAT = _X_tr.shape

print(f"\nTraining nodes : {N_TR:,}  "
      f"(fraud: {(_y_tr==1).sum():,} | non-fraud: {(_y_tr==0).sum():,})")
print(f"Test nodes     : {len(_y_te):,}  (fraud: {(_y_te==1).sum()})")

# ── 2. Build k-NN graph → edge index tensors ──────────────────────────────────
K_NEIGH = 10
print(f"\nBuilding k-NN similarity graph (k={K_NEIGH}) on {N_TR:,} training nodes …")

A_knn = kneighbors_graph(_X_tr, n_neighbors=K_NEIGH,
                          mode='connectivity', include_self=False, n_jobs=-1)
A_sym = A_knn + A_knn.T
A_sym.data[:] = 1.0
A_coo = A_sym.tocoo()

_ei_src = torch.tensor(A_coo.row.astype('int64'), dtype=torch.long)
_ei_dst = torch.tensor(A_coo.col.astype('int64'), dtype=torch.long)
_deg    = torch.bincount(_ei_src, minlength=N_TR).float().clamp(min=1)
print(f"  Undirected edges : {_ei_src.size(0):,}")

# ── 3. Mean-aggregation helper (scatter_add_ — CPU / CUDA / MPS safe) ─────────
def _mean_agg(H, src, dst, deg):
    dev = H.device
    out = torch.zeros(deg.size(0), H.size(1), device=dev, dtype=H.dtype)
    out.scatter_add_(0, src.unsqueeze(1).expand(-1, H.size(1)).to(dev), H[dst.to(dev)])
    return out / deg.to(dev).unsqueeze(1)

# ── 4. GraphSAGE model ────────────────────────────────────────────────────────
class GraphSAGE(nn.Module):
    def __init__(self, in_ch, hid_ch=64, out_ch=2, drop=0.3):
        super().__init__()
        self.fc1s = nn.Linear(in_ch,  hid_ch)
        self.fc1n = nn.Linear(in_ch,  hid_ch, bias=False)
        self.fc2s = nn.Linear(hid_ch, out_ch)
        self.fc2n = nn.Linear(hid_ch, out_ch, bias=False)
        self.drop = drop

    def forward(self, x, src, dst, deg):
        h1 = F.relu(self.fc1s(x) + self.fc1n(_mean_agg(x, src, dst, deg)))
        h1 = F.dropout(h1, p=self.drop, training=self.training)
        return self.fc2s(h1) + self.fc2n(_mean_agg(h1, src, dst, deg))

sage = GraphSAGE(N_FEAT, hid_ch=64, out_ch=2).to(_dev)
print(f"\nGraphSAGE : {N_FEAT} → SAGEConv(64) → ReLU/Drop(0.3) → SAGEConv(2)")
print(f"  Parameters : {sum(p.numel() for p in sage.parameters()):,}")

# ── 5. Training (manual Adam + LR patience) ───────────────────────────────────
X_tr_t = torch.from_numpy(_X_tr).to(_dev)
y_tr_t = torch.from_numpy(_y_tr).to(_dev)
src_t  = _ei_src.to(_dev);  dst_t = _ei_dst.to(_dev);  deg_t = _deg.to(_dev)

_w1  = float((_y_tr==0).sum()) / max(float((_y_tr==1).sum()), 1)
crit = nn.CrossEntropyLoss(weight=torch.tensor([1.0, _w1]).to(_dev))
opt  = _Adam(sage.parameters(), lr=0.005, weight_decay=1e-4)

MAX_EP = 200; PAT = 25; PAT_LR = 10
best_loss = float('inf'); wait = 0; wait_lr = 0; best_lr_loss = float('inf'); best_st = None

print(f"\nTraining ({MAX_EP} max epochs, patience={PAT}) …")
sage.train()
for ep in range(1, MAX_EP + 1):
    opt.zero_grad()
    loss = crit(sage(X_tr_t, src_t, dst_t, deg_t), y_tr_t)
    loss.backward()
    opt.step()

    lv = loss.item()
    if lv < best_lr_loss: best_lr_loss = lv; wait_lr = 0
    else:
        wait_lr += 1
        if wait_lr >= PAT_LR: opt.lr *= 0.5; wait_lr = 0; best_lr_loss = lv

    if lv < best_loss:
        best_loss = lv; wait = 0
        best_st = {k: v.clone() for k, v in sage.state_dict().items()}
    else:
        wait += 1
        if wait >= PAT:
            print(f"  Early stop at epoch {ep}  (best loss={best_loss:.5f})"); break
    if ep % 50 == 0 or ep == 1:
        print(f"  Epoch {ep:3d}  loss={lv:.5f}  lr={opt.lr:.5f}")

sage.load_state_dict(best_st)
print(f"  ✓ Best training loss = {best_loss:.5f}")

# ── 6. Precompute layer-1 embeddings ──────────────────────────────────────────
sage.eval()
with torch.no_grad():
    h1_tr_t = F.relu(sage.fc1s(X_tr_t) + sage.fc1n(_mean_agg(X_tr_t, src_t, dst_t, deg_t)))
h1_tr_np = h1_tr_t.cpu().numpy()
print(f"\nPrecomputed layer-1 train embeddings : {h1_tr_np.shape}")

# ── 7. Inductive inference ────────────────────────────────────────────────────
nn_idx = NearestNeighbors(n_neighbors=K_NEIGH, n_jobs=-1).fit(_X_tr)

def _infer(X_query, batch=4096):
    n = len(X_query);  _, nbr = nn_idx.kneighbors(X_query)
    out = []
    with torch.no_grad():
        for s in range(0, n, batch):
            e   = min(s + batch, n)
            xq  = torch.from_numpy(X_query[s:e]).to(_dev)
            idx = nbr[s:e]
            xn  = torch.from_numpy(_X_tr[idx].mean(axis=1).astype('float32')).to(_dev)
            h1  = F.relu(sage.fc1s(xq) + sage.fc1n(xn))
            h1n = torch.from_numpy(h1_tr_np[idx].mean(axis=1).astype('float32')).to(_dev)
            h2  = sage.fc2s(h1) + sage.fc2n(h1n)
            out.append(F.softmax(h2, dim=1)[:, 1].cpu().numpy())
    return np.concatenate(out)

# ── 8. Balanced val set for threshold selection ────────────────────────────────
_rng_v      = np.random.RandomState(RANDOM_STATE + 7)
_fraud_idx  = np.where(_yo == 1)[0]
_nf_idx     = np.where(_yo == 0)[0]
_sel_nf     = _rng_v.choice(_nf_idx, size=len(_fraud_idx), replace=False)
_bal_idx    = np.concatenate([_fraud_idx, _sel_nf])
_Xbal       = _Xo[_bal_idx]
_ybal       = _yo[_bal_idx]
print(f"\nBalanced val set : {len(_ybal):,} nodes  "
      f"(fraud: {(_ybal==1).sum()}  |  non-fraud: {(_ybal==0).sum()})")

print(f"Inductive inference — balanced val ({len(_ybal):,} nodes) …")
_val_raw_bal = _infer(_Xbal)
print(f"Inductive inference — test set ({len(_y_te):,} nodes) …")
_test_raw    = _infer(_X_te)

# ── 9. Youden's J threshold on balanced val ───────────────────────────────────
_fpr_v, _tpr_v, _thr_v = roc_curve(_ybal, _val_raw_bal)
_bj      = int(np.argmax(_tpr_v - _fpr_v))
_opt_thr = float(_thr_v[_bj])

# Clamp to a sensible range if ROC is degenerate
if not np.isfinite(_opt_thr):
    _opt_thr = float(np.percentile(_test_raw, 90))
    print(f"  (degenerate ROC — using 90th-percentile fallback threshold)")

print(f"\nYouden's J threshold (balanced val) : {_opt_thr:.4f}")
print(f"  Val TPR (recall) : {_tpr_v[_bj]:.4f}")
print(f"  Val FPR          : {_fpr_v[_bj]:.4f}")

_fp = _test_raw[_y_te == 1]
print(f"\nRaw fraud probs (test) — "
      f"min:{_fp.min():.4f}  max:{_fp.max():.4f}  mean:{_fp.mean():.4f}")

# ── 10. Evaluate on held-out test set ─────────────────────────────────────────
results_list[:] = [r for r in results_list if r.get('Model') != 'GraphSAGE (GNN)']

_pred = (_test_raw >= _opt_thr).astype(int)
sage_results = evaluate_model(y_test, _pred, _test_raw, "GraphSAGE (GNN)")
results_list.append(sage_results)
print("\n✓ GraphSAGE evaluation complete!")


### 11.8 FNN — Tabular Residual Network (TabResNet)

In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# Section 11.8  FNN — Tabular Residual Network (TabResNet)
# ═══════════════════════════════════════════════════════════════════════════════
#
# Why TabResNet vs. the existing MLP (11.6)?
#   • Skip connections prevent vanishing gradients in deeper architectures.
#   • Batch normalisation stabilises training on tabular fraud data.
#   • Class-weighted loss directly addresses the ~0.017% fraud imbalance.
#
# Class-distribution policy:
#   Training, calibration, and threshold selection all use the ORIGINAL
#   class distribution (~0.017% fraud), NOT the SMOTE-balanced set.
#   Reason: training on y_train_balanced makes class_weight ≈ 1.0 (both
#   classes equal in SMOTE data), so the weighted loss provides zero
#   fraud correction.  Using the original imbalanced data lets the
#   neg/pos ratio (~5 000–6 000×) drive the loss weighting correctly,
#   and threshold selection on the same real distribution produces a
#   threshold that is calibrated to the true operating environment.
# ═══════════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_curve
from sklearn.model_selection import train_test_split

print("=" * 60)
print("Section 11.8  FNN — Tabular Residual Network (TabResNet)")
print("=" * 60)

# ── 0. Device ─────────────────────────────────────────────────────────────────
_rn_dev = (torch.device("mps")  if torch.backends.mps.is_available() else
            torch.device("cuda") if torch.cuda.is_available() else
            torch.device("cpu"))
print(f"  Device : {_rn_dev}")

# ── 1. Scaled original training data (original class distribution) ─────────────
if 'X_train_orig_scaled' not in dir():
    X_train_orig_scaled = pd.DataFrame(
        scaler.transform(X_train), columns=feature_names, index=X_train.index
    )

# ── 2. Minimal Adam (sidesteps torch._dynamo import issues) ───────────────────
if '_Adam' not in dir():
    class _Adam:
        """Pure-Python Adam; sidesteps torch._dynamo import in torch.optim."""
        def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0):
            self.params = [p for p in params if p.requires_grad]
            self.lr = lr; self.b1, self.b2 = betas
            self.eps = eps; self.wd = weight_decay; self.t = 0
            self.m = [torch.zeros_like(p.data) for p in self.params]
            self.v = [torch.zeros_like(p.data) for p in self.params]
        def zero_grad(self):
            for p in self.params:
                if p.grad is not None: p.grad.detach_(); p.grad.zero_()
        def step(self):
            self.t += 1
            bc1, bc2 = 1 - self.b1 ** self.t, 1 - self.b2 ** self.t
            for i, p in enumerate(self.params):
                if p.grad is None: continue
                g = p.grad.data
                if self.wd: g = g.add(p.data, alpha=self.wd)
                self.m[i].mul_(self.b1).add_(g, alpha=1 - self.b1)
                self.v[i].mul_(self.b2).addcmul_(g, g, value=1 - self.b2)
                p.data.addcdiv_(self.m[i] / bc1, (self.v[i] / bc2).sqrt().add_(self.eps),
                                value=-self.lr)

# ── 3. Tensor helpers ──────────────────────────────────────────────────────────
def _rn_t32(a):
    return torch.from_numpy(
        (a.values if hasattr(a, 'values') else np.array(a)).astype('float32'))

def _rn_t64(a):
    return torch.from_numpy(
        (a.values if hasattr(a, 'values') else np.array(a)).astype('int64'))

# Use original unbalanced training data — preserves true ~0.017% fraud rate
Xtr_rn = _rn_t32(X_train_orig_scaled)
ytr_rn = _rn_t64(y_train)
Xte_rn = _rn_t32(X_test_scaled)
_n_in  = Xtr_rn.shape[1]

_n_neg_rn = int((ytr_rn == 0).sum())
_n_pos_rn = int((ytr_rn == 1).sum())
print(f"\nTraining samples : {len(ytr_rn):,}  "
      f"(fraud={_n_pos_rn:,} | non-fraud={_n_neg_rn:,}  "
      f"rate={100.0*_n_pos_rn/len(ytr_rn):.4f}%)")
print(f"Test samples     : {len(y_test):,}")
print(f"Input features   : {_n_in}")

# ── 4. Pre-activation Residual Block ──────────────────────────────────────────
class _ResBlock(nn.Module):
    """BN → ReLU → Linear → Dropout → BN → ReLU → Linear  +  skip connection."""
    def __init__(self, dim, drop=0.3):
        super().__init__()
        self.bn1  = nn.BatchNorm1d(dim)
        self.lin1 = nn.Linear(dim, dim)
        self.bn2  = nn.BatchNorm1d(dim)
        self.lin2 = nn.Linear(dim, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        h = self.drop(self.lin1(F.relu(self.bn1(x))))
        h = self.lin2(F.relu(self.bn2(h)))
        return x + h                         # residual skip

# ── 5. Tabular ResNet ──────────────────────────────────────────────────────────
class _TabResNet(nn.Module):
    def __init__(self, in_dim, hidden=256, n_blocks=3, drop=0.3):
        super().__init__()
        self.stem   = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
        )
        self.blocks = nn.Sequential(
            *[_ResBlock(hidden, drop) for _ in range(n_blocks)]
        )
        self.head   = nn.Linear(hidden, 2)

    def forward(self, x):
        return self.head(self.blocks(self.stem(x)))

tab_resnet = _TabResNet(in_dim=_n_in, hidden=256, n_blocks=3, drop=0.3).to(_rn_dev)
print(f"\nTabResNet : {_n_in} → stem(256,BN,ReLU) → 3×ResBlock(256,drop=0.3) → head(2)")
print(f"  Trainable parameters : {sum(p.numel() for p in tab_resnet.parameters()):,}")

# ── 6. Class-weighted loss (weight reflects true fraud prevalence) ─────────────
# With y_train_balanced the ratio ≈ 1.0 (no effect). With original y_train
# the neg/pos ratio correctly penalises missed frauds by ~5000–6000×.
_w_rn    = float(_n_neg_rn) / max(float(_n_pos_rn), 1)
_crit_rn = nn.CrossEntropyLoss(weight=torch.tensor([1.0, _w_rn], device=_rn_dev))
print(f"\nClass weight for fraud class : {_w_rn:.1f}  (neg/pos from original data)")

# ── 7. Mini-batch DataLoader ───────────────────────────────────────────────────
_loader_rn = DataLoader(
    TensorDataset(Xtr_rn, ytr_rn),
    batch_size=512, shuffle=True, drop_last=False
)

_opt_rn = _Adam(tab_resnet.parameters(), lr=1e-3, weight_decay=1e-4)

# ── 8. Training with early stopping + LR patience decay ───────────────────────
_MAX_EP_RN    = 150
_PAT_RN       = 20
_PAT_LR_RN    = 8
_best_rn_loss = float('inf')
_wait_rn      = 0
_wait_lr_rn   = 0
_best_lr_rn   = float('inf')
_best_st_rn   = None

print(f"\nTraining ({_MAX_EP_RN} max epochs, patience={_PAT_RN}, batch=512) …")
tab_resnet.train()
for _ep_rn in range(1, _MAX_EP_RN + 1):
    _ep_loss = 0.0
    for _Xb, _yb in _loader_rn:
        _Xb, _yb = _Xb.to(_rn_dev), _yb.to(_rn_dev)
        _opt_rn.zero_grad()
        _l = _crit_rn(tab_resnet(_Xb), _yb)
        _l.backward()
        _opt_rn.step()
        _ep_loss += _l.item() * len(_yb)
    _ep_loss /= len(ytr_rn)

    if _ep_loss < _best_lr_rn:
        _best_lr_rn = _ep_loss; _wait_lr_rn = 0
    else:
        _wait_lr_rn += 1
        if _wait_lr_rn >= _PAT_LR_RN:
            _opt_rn.lr *= 0.5; _wait_lr_rn = 0; _best_lr_rn = _ep_loss

    if _ep_loss < _best_rn_loss:
        _best_rn_loss = _ep_loss; _wait_rn = 0
        _best_st_rn   = {k: v.clone() for k, v in tab_resnet.state_dict().items()}
    else:
        _wait_rn += 1
        if _wait_rn >= _PAT_RN:
            print(f"  Early stop at epoch {_ep_rn}  (best loss={_best_rn_loss:.5f})")
            break

    if _ep_rn % 30 == 0 or _ep_rn == 1:
        print(f"  Epoch {_ep_rn:3d}  loss={_ep_loss:.5f}  lr={_opt_rn.lr:.6f}")

tab_resnet.load_state_dict(_best_st_rn)
print(f"  ✓ Best training loss = {_best_rn_loss:.5f}")

# ── 9. Threshold selection on original-distribution val split ─────────────────
# A 20% stratified split of original training data preserves the real fraud
# rate (~0.017%), so Youden's J finds a threshold calibrated to true prevalence,
# not to an artificial 50/50 balanced set.
_, _Xval_rn_np, _, _yval_rn = train_test_split(
    np.array(X_train_orig_scaled, dtype='float32'),
    np.array(y_train, dtype='int64'),
    test_size=0.20, random_state=RANDOM_STATE, stratify=y_train
)
_Xval_rn_t = torch.from_numpy(_Xval_rn_np).to(_rn_dev)

print(f"\nVal set (original distribution) : {len(_yval_rn):,}  "
      f"(fraud: {(_yval_rn==1).sum()} | non-fraud: {(_yval_rn==0).sum()}  "
      f"rate={100.0*(_yval_rn==1).mean():.4f}%)")

tab_resnet.eval()
with torch.no_grad():
    _val_proba_rn = F.softmax(tab_resnet(_Xval_rn_t), dim=1)[:, 1].cpu().numpy()

_fpr_rn, _tpr_rn, _thr_rn = roc_curve(_yval_rn, _val_proba_rn)
_best_j_rn  = int(np.argmax(_tpr_rn - _fpr_rn))
_opt_thr_rn = float(_thr_rn[_best_j_rn])

print(f"\nYouden's J threshold (original val) : {_opt_thr_rn:.4f}")
print(f"  Val TPR (recall) : {_tpr_rn[_best_j_rn]:.4f}")
print(f"  Val FPR          : {_fpr_rn[_best_j_rn]:.4f}")

# ── 10. Evaluate on held-out test set ─────────────────────────────────────────
with torch.no_grad():
    _te_proba_rn = F.softmax(
        tab_resnet(Xte_rn.to(_rn_dev)), dim=1)[:, 1].cpu().numpy()

_te_pred_rn   = (_te_proba_rn >= _opt_thr_rn).astype(int)
resnet_results = evaluate_model(y_test, _te_pred_rn, _te_proba_rn, "TabResNet (FNN)")
results_list   = [r for r in results_list if r.get('model') != 'TabResNet (FNN)']
results_list.append(resnet_results)
print("\n✓ Tabular ResNet evaluation complete!")


### 11.9 FNN — Deep Neural Network (DNN)

In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# Section 11.9  FNN — Deep Neural Network (DNN)
# ═══════════════════════════════════════════════════════════════════════════════
#
# Architecture: 5 hidden layers (512→256→128→64→32) — deeper than MLP (11.6)
# and wider than TabResNet (11.8, skip-connection blocks).
# Key differences from existing FNN models:
#   • No skip connections (straight deep path vs TabResNet)
#   • PReLU activations — learnable negative slope per layer
#   • Label-smoothed cross-entropy — softens hard 0/1 targets, reducing
#     over-confidence on the heavily imbalanced Medicare data
#
# Class-distribution policy (same as Section 11.8):
#   Training, calibration, and threshold selection all use the ORIGINAL
#   class distribution (~0.017% fraud), NOT the SMOTE-balanced set.
#   Reason: training on y_train_balanced makes class_weight ≈ 1.0 (both
#   classes equal in SMOTE data), so the weighted loss provides zero
#   fraud correction.  Using the original imbalanced data lets the true
#   neg/pos ratio drive the loss correctly, and the threshold found on
#   the original-distribution val set is calibrated to real prevalence.
# ═══════════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_curve
from sklearn.model_selection import train_test_split

print("=" * 60)
print("Section 11.9  FNN — Deep Neural Network (DNN)")
print("=" * 60)

# ── 0. Device ─────────────────────────────────────────────────────────────────
_dnn_dev = (torch.device("mps")  if torch.backends.mps.is_available() else
             torch.device("cuda") if torch.cuda.is_available() else
             torch.device("cpu"))
print(f"  Device : {_dnn_dev}")

# ── 1. Scaled original training data (original class distribution) ─────────────
if 'X_train_orig_scaled' not in dir():
    X_train_orig_scaled = pd.DataFrame(
        scaler.transform(X_train), columns=feature_names, index=X_train.index
    )

# ── 2. Minimal Adam (sidesteps torch._dynamo import issues) ───────────────────
if '_Adam' not in dir():
    class _Adam:
        """Pure-Python Adam; sidesteps torch._dynamo import in torch.optim."""
        def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0):
            self.params = [p for p in params if p.requires_grad]
            self.lr = lr; self.b1, self.b2 = betas
            self.eps = eps; self.wd = weight_decay; self.t = 0
            self.m = [torch.zeros_like(p.data) for p in self.params]
            self.v = [torch.zeros_like(p.data) for p in self.params]
        def zero_grad(self):
            for p in self.params:
                if p.grad is not None: p.grad.detach_(); p.grad.zero_()
        def step(self):
            self.t += 1
            bc1, bc2 = 1 - self.b1 ** self.t, 1 - self.b2 ** self.t
            for i, p in enumerate(self.params):
                if p.grad is None: continue
                g = p.grad.data
                if self.wd: g = g.add(p.data, alpha=self.wd)
                self.m[i].mul_(self.b1).add_(g, alpha=1 - self.b1)
                self.v[i].mul_(self.b2).addcmul_(g, g, value=1 - self.b2)
                p.data.addcdiv_(self.m[i] / bc1, (self.v[i] / bc2).sqrt().add_(self.eps),
                                value=-self.lr)

# ── 3. Data tensors (original unbalanced distribution) ────────────────────────
def _dnn_t32(a):
    return torch.from_numpy(
        (a.values if hasattr(a, 'values') else np.array(a)).astype('float32'))

def _dnn_t64(a):
    return torch.from_numpy(
        (a.values if hasattr(a, 'values') else np.array(a)).astype('int64'))

# Use original unbalanced training data — preserves ~0.017% fraud rate
Xtr_dnn = _dnn_t32(X_train_orig_scaled)
ytr_dnn = _dnn_t64(y_train)
Xte_dnn = _dnn_t32(X_test_scaled)
_n_in_dnn = Xtr_dnn.shape[1]

_n_neg_dnn = int((ytr_dnn == 0).sum())
_n_pos_dnn = int((ytr_dnn == 1).sum())
print(f"\nTraining samples : {len(ytr_dnn):,}  "
      f"(fraud={_n_pos_dnn:,} | non-fraud={_n_neg_dnn:,}  "
      f"rate={100.0*_n_pos_dnn/len(ytr_dnn):.4f}%)")
print(f"Test samples     : {len(y_test):,}")
print(f"Input features   : {_n_in_dnn}")

# ── 4. DNN model: 5 hidden BN→PReLU→Dropout layers ───────────────────────────
class _DNN(nn.Module):
    """Straight deep FNN: Linear → BN → PReLU → Dropout (×5 layers) → head."""
    def __init__(self, in_dim, layers=(512, 256, 128, 64, 32), drop=0.4):
        super().__init__()
        dims = [in_dim] + list(layers)
        blocks = []
        for i in range(len(layers)):
            blocks += [
                nn.Linear(dims[i], dims[i+1]),
                nn.BatchNorm1d(dims[i+1]),
                nn.PReLU(),
                nn.Dropout(drop),
            ]
        self.body = nn.Sequential(*blocks)
        self.head = nn.Linear(dims[-1], 2)

    def forward(self, x):
        return self.head(self.body(x))

dnn_model = _DNN(in_dim=_n_in_dnn, layers=(512, 256, 128, 64, 32), drop=0.4).to(_dnn_dev)
print(f"\nDNN : {_n_in_dnn}→512→256→128→64→32→2  (BN+PReLU+Drop(0.4) each layer)")
print(f"  Trainable parameters : {sum(p.numel() for p in dnn_model.parameters()):,}")

# ── 5. Label-smoothed class-weighted loss (weight from true class distribution) ─
# With y_train_balanced the neg/pos ratio ≈ 1.0 (no fraud correction).
# With original y_train the ratio correctly penalises missed fraud by ~5000–6000×.
_w_dnn  = float(_n_neg_dnn) / max(float(_n_pos_dnn), 1)
_SMOOTH = 0.05
print(f"\nClass weight for fraud class : {_w_dnn:.1f}  (neg/pos from original data)")

class _SmoothedWeightedCE(nn.Module):
    """CrossEntropy + label smoothing + per-class weight (device-safe)."""
    def __init__(self, weight, smoothing=0.05):
        super().__init__()
        self._w_cpu    = weight.cpu()
        self.smoothing = smoothing

    def forward(self, logits, targets):
        dev       = logits.device
        w         = self._w_cpu.to(dev)
        log_probs = F.log_softmax(logits, dim=-1)
        nll       = F.nll_loss(log_probs, targets, weight=w)
        smooth    = -log_probs.mean(dim=-1)
        wt        = self._w_cpu[targets.cpu()].to(dev)
        smooth    = (smooth * wt).mean()
        return (1.0 - self.smoothing) * nll + self.smoothing * smooth

_crit_dnn = _SmoothedWeightedCE(
    weight=torch.tensor([1.0, _w_dnn]), smoothing=_SMOOTH)

# ── 6. DataLoader ──────────────────────────────────────────────────────────────
_loader_dnn = DataLoader(
    TensorDataset(Xtr_dnn, ytr_dnn),
    batch_size=512, shuffle=True, drop_last=False
)

_opt_dnn = _Adam(dnn_model.parameters(), lr=1e-3, weight_decay=1e-4)

# ── 7. Training with early stopping + LR patience decay ───────────────────────
_MAX_EP_DNN    = 150
_PAT_DNN       = 20
_PAT_LR_DNN    = 8
_best_dnn_loss = float('inf')
_wait_dnn      = 0
_wait_lr_dnn   = 0
_best_lr_dnn   = float('inf')
_best_st_dnn   = None

print(f"\nTraining ({_MAX_EP_DNN} max epochs, patience={_PAT_DNN}, batch=512) …")
dnn_model.train()
for _ep_dnn in range(1, _MAX_EP_DNN + 1):
    _ep_loss_dnn = 0.0
    for _Xb, _yb in _loader_dnn:
        _Xb, _yb = _Xb.to(_dnn_dev), _yb.to(_dnn_dev)
        _opt_dnn.zero_grad()
        _l = _crit_dnn(dnn_model(_Xb), _yb)
        _l.backward()
        _opt_dnn.step()
        _ep_loss_dnn += _l.item() * len(_yb)
    _ep_loss_dnn /= len(ytr_dnn)

    if _ep_loss_dnn < _best_lr_dnn:
        _best_lr_dnn = _ep_loss_dnn; _wait_lr_dnn = 0
    else:
        _wait_lr_dnn += 1
        if _wait_lr_dnn >= _PAT_LR_DNN:
            _opt_dnn.lr *= 0.5; _wait_lr_dnn = 0; _best_lr_dnn = _ep_loss_dnn

    if _ep_loss_dnn < _best_dnn_loss:
        _best_dnn_loss = _ep_loss_dnn; _wait_dnn = 0
        _best_st_dnn   = {k: v.clone() for k, v in dnn_model.state_dict().items()}
    else:
        _wait_dnn += 1
        if _wait_dnn >= _PAT_DNN:
            print(f"  Early stop at epoch {_ep_dnn}  (best loss={_best_dnn_loss:.5f})")
            break

    if _ep_dnn % 30 == 0 or _ep_dnn == 1:
        print(f"  Epoch {_ep_dnn:3d}  loss={_ep_loss_dnn:.5f}  lr={_opt_dnn.lr:.6f}")

dnn_model.load_state_dict(_best_st_dnn)
print(f"  ✓ Best training loss = {_best_dnn_loss:.5f}")

# ── 8. Threshold selection on original-distribution val split ─────────────────
# A 20% stratified split of original training data preserves the real fraud
# rate (~0.017%), so Youden's J finds a threshold calibrated to true prevalence,
# not to an artificial 50/50 balanced set.
_, _Xval_dnn_np, _, _yval_dnn = train_test_split(
    np.array(X_train_orig_scaled, dtype='float32'),
    np.array(y_train, dtype='int64'),
    test_size=0.20, random_state=RANDOM_STATE, stratify=y_train
)
_Xval_dnn_t = torch.from_numpy(_Xval_dnn_np).to(_dnn_dev)

print(f"\nVal set (original distribution) : {len(_yval_dnn):,}  "
      f"(fraud: {(_yval_dnn==1).sum()} | non-fraud: {(_yval_dnn==0).sum()}  "
      f"rate={100.0*(_yval_dnn==1).mean():.4f}%)")

dnn_model.eval()
with torch.no_grad():
    _val_proba_dnn = F.softmax(dnn_model(_Xval_dnn_t), dim=1)[:, 1].cpu().numpy()

_fpr_dnn, _tpr_dnn, _thr_dnn = roc_curve(_yval_dnn, _val_proba_dnn)
_best_j_dnn   = int(np.argmax(_tpr_dnn - _fpr_dnn))
_opt_thr_dnn  = float(_thr_dnn[_best_j_dnn])

print(f"\nYouden's J threshold (original val) : {_opt_thr_dnn:.4f}")
print(f"  Val TPR (recall) : {_tpr_dnn[_best_j_dnn]:.4f}")
print(f"  Val FPR          : {_fpr_dnn[_best_j_dnn]:.4f}")

# ── 9. Evaluate on held-out test set ──────────────────────────────────────────
with torch.no_grad():
    _te_proba_dnn = F.softmax(
        dnn_model(Xte_dnn.to(_dnn_dev)), dim=1)[:, 1].cpu().numpy()

_te_pred_dnn = (_te_proba_dnn >= _opt_thr_dnn).astype(int)
dnn_results  = evaluate_model(y_test, _te_pred_dnn, _te_proba_dnn, "DNN (FNN)")
results_list = [r for r in results_list if r.get('model') != 'DNN (FNN)']
results_list.append(dnn_results)
print("\n✓ DNN evaluation complete!")



### 17.3 Hyperparameter Search Results


| Symbol | Strategy | Description |
|--------|----------|-------------|
| **SMOTE-ENN ↓** | Two-step subsample resampling | RandomUnderSampler (majority → 30,000) → SMOTE-ENN (`sampling_strategy=0.10`, `k_neighbors=5`) → **~29,535 rows** (~8.85:1 ratio); 3,000 fraud (2,784 synthetic + 216 real). Applied before scaling; test set untouched. |

---

#### Run 1 — 22-Feature Set (Original, RandomizedSearch)

| Model | Section | Search Method | # Fits | Training Data (CV phase) | Calibration | Best Key Hyperparameters | CV ROC-AUC | Test ROC-AUC | Test Recall | Fraud Detected (54 total) | Youden Thr. |
|-------|---------|--------------|--------|--------------------------|-------------|--------------------------|------------|-------------|-------------|--------------------------|-------------|
| **Random Forest** | 11.1 | RandomizedSearch (15 iter × 3-fold) | 45 | 23,628 rows (80% of SMOTE-ENN set) | Isotonic | `n_estimators=100`, `max_depth=8`, `max_features=log2`, `max_samples=0.5`, `min_samples_leaf=4` | 0.5771 | 0.6626 | 25.9% | 14 | 0.0002 |
| **XGBoost** | 11.2 | RandomizedSearch (15 iter × 3-fold) | 45 | 23,628 rows (SMOTE-ENN) | Isotonic | `n_estimators=300`, `max_depth=3`, `lr=0.05`, `reg_alpha=0.5`, `reg_lambda=1.0` | 0.7300 | 0.6194 | 31.5% | 17 | 0.0002 |
| **LightGBM** | 11.3 | RandomizedSearch (15 iter × 3-fold) | 45 | 23,628 rows (80% of SMOTE-ENN set) | Isotonic | `n_estimators=300`, `num_leaves=15`, `max_depth=4`, `lr=0.05`, `min_child_samples=50`, `lambda_l2=1.0` | 0.7211 | 0.5677 | 25.9% | 14 | 0.0003 |
| **HistGradBoost** | 11.4 | RandomizedSearch (15 iter × 3-fold) | 45 | 23,628 rows (80% of SMOTE-ENN set) | Isotonic | `max_iter=200`, `max_leaf_nodes=15`, `max_depth=None`, `lr=0.05`, `min_samples_leaf=20`, `l2=0.0` | 0.7218 | 0.5866 | 33.3% | 18 | 0.0002 |
| **SVM (LinearSVC)** | 11.5 | GridSearch (6 C values × 3-fold) | 18 | 23,628 rows (80% of SMOTE-ENN set) | Platt (sigmoid) | `C=0.01` | 0.7183 | **0.7059** | **75.9%** | **41** | 0.0002 |
| **MLP (Neural Net)** | 11.6 | RandomizedSearch (10 iter × 3-fold) | 30 | 20,000 rows (SMOTE-ENN subsample) | Platt (sigmoid) | `hidden=(256,128,64)`, `lr=0.0005`, `alpha=0.001`, `batch=256` | 0.9608 | 0.6226 | 27.8% | 15 | ~0.09 |
| **GraphSAGE (GNN)** | 11.7 | Fixed architecture — no CV search | — | 30,015 nodes (SMOTE-ENN) | Balanced 50/50 val + Youden's J | SAGEConv(64)×2, k-NN k=10, 200 epochs, dropout=0.3 | — | 0.6340 | 38.9% | 21 | ~0.37 |

---

#### Run 2 — 39-Feature Set (Updated, March 2026, RandomizedSearch)

| Model | Section | Search Method | # Fits | Training Data (CV phase) | Calibration | Best Key Hyperparameters | CV ROC-AUC | Test ROC-AUC | Test Recall | Fraud Detected (54 total) | Youden Thr. |
|-------|---------|--------------|--------|--------------------------|-------------|--------------------------|------------|-------------|-------------|--------------------------|-------------|
| **Logistic Regression** | 10 | No search (fixed defaults) | — | 29,535 rows (full SMOTE-ENN set) | None | `max_iter=1000`, `C=1.0` | — | 0.7199 | **64.8%** | **35** | 0.50 (default) |
| **Random Forest** | 11.1 | RandomizedSearch (15 iter × 3-fold) | 45 | 23,628 rows (80% of SMOTE-ENN set) | Isotonic | `n_estimators=100`, `max_depth=8`, `max_features=log2`, `max_samples=0.5`, `min_samples_leaf=4`, `min_samples_split=20` | 0.5572 | 0.6850 | 31.5% | 17 | 0.0002 |
| **XGBoost** | 11.2 | RandomizedSearch (15 iter × 3-fold) | 45 | 23,628 rows (SMOTE-ENN) | Isotonic | `n_estimators=100`, `max_depth=3`, `lr=0.05`, `reg_alpha=0.1`, `reg_lambda=2.0`, `gamma=0.3` | 0.7389 | **0.7253** | 40.7% | 22 | 0.0003 |
| **LightGBM** | 11.3 | RandomizedSearch (15 iter × 3-fold) | 45 | 23,628 rows (80% of SMOTE-ENN set) | Isotonic | `n_estimators=300`, `num_leaves=15`, `max_depth=4`, `lr=0.05`, `min_child_samples=50`, `lambda_l2=1.0` | 0.7301 | 0.5623 | 14.8% | 8 | 0.0002 |
| **HistGradBoost** | 11.4 | RandomizedSearch (15 iter × 3-fold) | 45 | 23,628 rows (80% of SMOTE-ENN set) | Isotonic | `max_iter=200`, `max_leaf_nodes=15`, `max_depth=8`, `lr=0.05`, `min_samples_leaf=50`, `l2=1.0` | 0.7277 | 0.5928 | 22.2% | 12 | 0.0002 |
| **SVM (LinearSVC)** | 11.5 | GridSearch (6 C values × 3-fold) | 18 | 23,628 rows (80% of SMOTE-ENN set) | Platt (sigmoid) | `C=0.01` | 0.7356 | 0.7292 | 53.7% | 29 | 0.0002 |
| **MLP (Neural Net)** | 11.6 | RandomizedSearch (10 iter × 3-fold) | 30 | 20,000 rows (SMOTE-ENN subsample) | Platt (sigmoid) | `hidden=(128,64,32)`, `lr=0.001`, `alpha=0.01`, `batch=256` | 0.9807 | 0.6302 | 16.7% | 9 | 0.0001 |
| **GraphSAGE (GNN)** | 11.7 | Fixed architecture — no CV search | — | 29,535 nodes (SMOTE-ENN) | Balanced 50/50 val + Youden's J | SAGEConv(64)×2, k-NN k=10, 200 epochs, dropout=0.3 | — | 0.6648 | 48.1% | 26 | 0.4518 |

---

#### Run 3 — Refined Hyperopt Search Spaces (March 2026, Hyperopt TPE)

> Search spaces were tightened to combat the extreme overfit seen in prior runs (CV AUC ~0.99 → Test AUC ~0.52). Depth caps, stricter regularisation floors, and subsampling constraints were applied. Calibration changed from `cv='prefit'` to `cv=3` (internal 3-fold). Logistic Regression and GraphSAGE are unchanged (no hyperparameter search).

| Model | Section | Search Method | # Fits | Training Data (CV phase) | Calibration | Best Key Hyperparameters | CV Score | Test ROC-AUC | Test Recall | Fraud Detected (54 total) | Youden Thr. |
|-------|---------|--------------|--------|--------------------------|-------------|--------------------------|----------|-------------|-------------|--------------------------|-------------|
| **Logistic Regression** | 10 | No search (unchanged) | — | 29,535 rows | None | `max_iter=1000`, `C=1.0` | — | 0.7199 | 64.8% | 35 | 0.50 |
| **Random Forest** | 11.1 | Hyperopt TPE (60 evals × 3-fold) | 180 | 23,628 rows (80% of SMOTE-ENN set) | Isotonic (cv=3) | `n_estimators=400`, `max_depth=8`, `max_features=0.5`, `max_samples=0.6`, `min_samples_leaf=6`, `min_samples_split=8` | 0.9575 (AUC) | **0.7630** | **55.6%** | **30** | 0.0934 |
| **Random Forest with F1 score** | 11.1 | Hyperopt TPE (60 evals × 3-fold, CV scoring=F1) | 180 | 23,628 rows (80% of SMOTE-ENN set) | Isotonic (cv=3) | `n_estimators=300`, `max_depth=8`, `max_features=0.5`, `max_samples=0.8`, `min_samples_leaf=8`, `min_samples_split=16` | 0.6983 (F1) | 0.7573 | 53.7% | 29 | 0.1170 |
| **XGBoost** | 11.2 | Hyperopt TPE (60 evals × 3-fold) | 180 | 23,628 rows (80% of SMOTE-ENN set) | Isotonic (cv=3) | `n_estimators=500`, `max_depth=5`, `lr=0.088`, `subsample=0.754`, `colsample=0.593`, `min_child_weight=5`, `gamma=1.065`, `reg_alpha=0.419`, `reg_lambda=1.689` | 0.9906 (AUC) | 0.7117 | 16.7% | 9 | 0.1233 |
| **LightGBM** | 11.3 | Hyperopt TPE (60 evals × 3-fold) | 180 | 23,628 rows (80% of SMOTE-ENN set) | Isotonic (cv=3) | `n_estimators=450`, `num_leaves=44`, `max_depth=-1`, `lr=0.119`, `min_child_samples=130`, `lambda_l1=0.018`, `lambda_l2=2.036`, `subsample=0.755` | 0.9958 (AUC) | 0.6829 | 11.1% | 6 | 0.1369 |
| **HistGradBoost** | 11.4 | Hyperopt TPE (60 evals × 3-fold) | 180 | 23,628 rows (80% of SMOTE-ENN set) | Isotonic (cv=3) | `max_iter=450`, `max_depth=15`, `lr=0.073`, `max_leaf_nodes=60`, `min_samples_leaf=125`, `l2=2.079`, `max_bins=64` | 0.9959 (AUC) | 0.6729 | 7.4% | 4 | 0.1565 |
| **SVM (LinearSVC)** | 11.5 | Hyperopt TPE (60 evals × 3-fold) | 180 | 23,628 rows (80% of SMOTE-ENN set) | Platt (cv=3) | `C=4.032` | **0.7992 (AUC)** | 0.7468 | **79.6%** | **43** | 0.0917 |
| **MLP (Neural Net)** | 11.6 | Hyperopt TPE (40 evals × 3-fold) | 120 | 20,000 rows (SMOTE-ENN subsample) | Platt (cv=3) | `hidden=(256,128,64)`, `alpha=0.000192`, `lr_init=0.00212`, `batch=256` | 0.9822 (AUC) | 0.6317 | 14.8% | 8 | 0.1007 |
| **GraphSAGE (GNN)** | 11.7 | Fixed architecture (unchanged) | — | 29,535 nodes (SMOTE-ENN) | Balanced 50/50 val + Youden's J | SAGEConv(64)×2, k-NN k=10, 200 epochs, dropout=0.3 | — | 0.6648 | 48.1% | 26 | 0.4518 |
| **TabResNet (FNN)** | 11.8 | Fixed architecture — no CV search | — | 29,535 rows (SMOTE-ENN) | Balanced 50/50 val + Youden's J | `stem(256,BN,ReLU) → 3×ResBlock(256,drop=0.3) → head(2)`, 150 max epochs, batch=512, class-weighted CE | — | 0.5242 | 48.1% | 26 | 0.2757 |
| **DNN (FNN)** | 11.9 | Fixed architecture — no CV search | — | 29,535 rows (SMOTE-ENN) | Balanced 50/50 val + Youden's J | `512→256→128→64→32`, PReLU+BN+Drop(0.4) per layer, label-smoothed class-weighted CE, 150 max epochs, batch=512 | — | 0.6735 | 44.4% | 24 | 0.7553 |

---

#### Run 1 → Run 2 Recall & AUC Comparison

| Model | Run 1 Recall (22-feat) | Run 1 AUC | Run 2 Recall (39-feat) | Run 2 AUC | Δ Recall | Δ AUC |
|-------|----------------------|-----------|----------------------|-----------|---------|-------|
| **Logistic Regression** | — | — | **64.8% (35/54)** | 0.7199 | ✨ NEW | — |
| **GraphSAGE (GNN)** | 38.9% (21/54) | 0.6340 | **48.1% (26/54)** | 0.6648 | 🟢 +9.3 pp | +0.031 |
| **XGBoost** | 31.5% (17/54) | 0.6194 | 40.7% (22/54) | **0.7253** | 🟢 +9.3 pp | +0.106 |
| **Random Forest** | 25.9% (14/54) | 0.6626 | 31.5% (17/54) | 0.6850 | 🟢 +5.6 pp | +0.023 |
| **HistGradBoost** | 33.3% (18/54) | 0.5866 | 22.2% (12/54) | 0.5928 | 🔴 −11.1 pp | +0.006 |
| **MLP (Neural Net)** | 27.8% (15/54) | 0.6226 | 16.7% (9/54) | 0.6302 | 🔴 −11.1 pp | +0.008 |
| **LightGBM** | 25.9% (14/54) | 0.5677 | 14.8% (8/54) | 0.5623 | 🔴 −11.1 pp | −0.005 |
| **SVM (LinearSVC)** | **75.9% (41/54)** | 0.7059 | 53.7% (29/54) | 0.7292 | 🔴 −22.2 pp | +0.023 |

---

#### Run 2 → Run 3 Recall & AUC Comparison (after refined Hyperopt search spaces)

| Model | Run 2 Recall | Run 2 AUC | Run 3 Recall | Run 3 AUC | Δ Recall | Δ AUC |
|-------|-------------|-----------|-------------|-----------|---------|-------|
| **SVM (LinearSVC)** | 53.7% (29/54) | 0.7292 | **79.6% (43/54)** | 0.7468 | 🟢 +25.9 pp | +0.018 |
| **Random Forest** | 31.5% (17/54) | 0.6850 | **55.6% (30/54)** | **0.7630** | 🟢 +24.1 pp | +0.078 |
| **Random Forest with F1 score** | 31.5% (17/54) | 0.6850 | 53.7% (29/54) | 0.7573 | 🟢 +22.2 pp | +0.091 |
| **Logistic Regression** | 64.8% (35/54) | 0.7199 | 64.8% (35/54) | 0.7199 | ➡ unchanged | — |
| **GraphSAGE (GNN)** | 48.1% (26/54) | 0.6648 | 48.1% (26/54) | 0.6648 | ➡ unchanged | — |
| **MLP (Neural Net)** | 16.7% (9/54) | 0.6302 | 14.8% (8/54) | 0.6317 | 🔴 −1.9 pp | +0.002 |
| **LightGBM** | 14.8% (8/54) | 0.5623 | 11.1% (6/54) | 0.6829 | 🔴 −3.7 pp | +0.121 |
| **HistGradBoost** | 22.2% (12/54) | 0.5928 | 7.4% (4/54) | 0.6729 | 🔴 −14.8 pp | +0.080 |
| **XGBoost** | 40.7% (22/54) | 0.7253 | 16.7% (9/54) | 0.7117 | 🔴 −24.0 pp | −0.014 |
